# Olist E-Commerce Data Analysis

## Phase 4: Advanced Analysis

This notebook focuses on advanced SQL analysis to uncover deeper business insights from the Olist E-Commerce dataset.

### Phase 4 Objectives

1. Month-over-Month Revenue Growth Analysis
2. Revenue Concentration and Pareto Analysis
3. Delivery Delay Severity vs Customer Satisfaction
4. RFM Customer Segmentation (subject to validation)
5. Seller Performance and Operational Analysis

In [5]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect("../data/olist_ecommerce.db")

print("SQLite database connection created successfully.")

SQLite database connection created successfully.


# Olist E-Commerce Data Analysis

## Phase 4: Advanced SQL Analysis

### Objective

This phase extends the core business analysis by applying advanced SQL techniques to uncover deeper insights from the Olist Brazilian E-Commerce dataset.

The analysis focuses on:

1. Month-over-Month Revenue Growth
2. Revenue Concentration and Pareto Analysis
3. Delivery Delay Severity and Customer Satisfaction
4. Customer Segmentation Validation and RFM Analysis
5. Seller Performance and Operational Analysis

### SQL Techniques Used

- Common Table Expressions (CTEs)
- Window Functions
- LAG()
- Running Totals
- CASE WHEN
- NTILE()
- Advanced Aggregations

**Note:** Advanced analyses will only be included when they provide meaningful business insights and are supported by the dataset.

## Database Verification

Before beginning the advanced analysis, verify that all required tables are available in the SQLite database.

In [6]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

tables

,name
0,customers
1,orders
2,order_items
3,payments
4,reviews
5,products
6,sellers
7,category_translation


## Data Schema Verification

Verify the structure of the key tables required for Phase 4 advanced analysis.

In [7]:
for table in ['orders', 'order_items', 'customers', 'reviews', 'sellers']:
    print(f"\n--- {table.upper()} ---")
    
    schema = pd.read_sql_query(
        f"PRAGMA table_info({table});",
        conn
    )
    
    print(schema[['name', 'type']].to_string(index=False))


--- ORDERS ---
                         name type
                     order_id TEXT
                  customer_id TEXT
                 order_status TEXT
     order_purchase_timestamp TEXT
            order_approved_at TEXT
 order_delivered_carrier_date TEXT
order_delivered_customer_date TEXT
order_estimated_delivery_date TEXT

--- ORDER_ITEMS ---
               name    type
           order_id    TEXT
      order_item_id INTEGER
         product_id    TEXT
          seller_id    TEXT
shipping_limit_date    TEXT
              price    REAL
      freight_value    REAL

--- CUSTOMERS ---
                      name    type
               customer_id    TEXT
        customer_unique_id    TEXT
  customer_zip_code_prefix INTEGER
             customer_city    TEXT
            customer_state    TEXT
customer_city_standardized    TEXT

--- REVIEWS ---
                   name    type
              review_id    TEXT
               order_id    TEXT
           review_score INTEGER
   review_comme

## Methodology Note

Month-over-Month growth should compare each month with the immediately preceding calendar month.

A continuous monthly timeline is therefore created before applying the `LAG()` window function. This prevents missing months from causing comparisons with an incorrect previous month.

Product revenue is defined consistently with the Phase 3 analysis as:

`SUM(order_items.price)`

In [5]:
query = """
WITH RECURSIVE months(order_month) AS (

    SELECT MIN(strftime('%Y-%m', order_purchase_timestamp))
    FROM orders

    UNION ALL

    SELECT strftime(
        '%Y-%m',
        date(order_month || '-01', '+1 month')
    )
    FROM months
    WHERE order_month < (
        SELECT MAX(strftime('%Y-%m', order_purchase_timestamp))
        FROM orders
    )
),

monthly_revenue AS (

    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
        ROUND(SUM(oi.price), 2) AS product_revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY strftime('%Y-%m', o.order_purchase_timestamp)
),

complete_monthly_revenue AS (

    SELECT
        m.order_month,
        COALESCE(mr.product_revenue, 0) AS product_revenue
    FROM months m
    LEFT JOIN monthly_revenue mr
        ON m.order_month = mr.order_month
),

revenue_with_previous AS (

    SELECT
        order_month,
        product_revenue,
        LAG(product_revenue) OVER (
            ORDER BY order_month
        ) AS previous_month_revenue
    FROM complete_monthly_revenue
)

SELECT
    order_month,
    product_revenue,
    previous_month_revenue,
    ROUND(
        CASE
            WHEN previous_month_revenue IS NULL
                 OR previous_month_revenue = 0
            THEN NULL

            ELSE (
                (product_revenue - previous_month_revenue)
                / previous_month_revenue
            ) * 100
        END,
        2
    ) AS mom_growth_percentage

FROM revenue_with_previous

ORDER BY order_month;
"""

mom_revenue = pd.read_sql_query(query, conn)

mom_revenue

,order_month,product_revenue,previous_month_revenue,mom_growth_percentage
0,2016-09,267.36,NaN,NaN
1,2016-10,49507.66,267.36,18417.23
2,2016-11,0.00,49507.66,-100.00
3,2016-12,10.90,0.00,NaN
4,2017-01,120312.87,10.90,1103687.80
5,2017-02,247303.02,120312.87,105.55
6,2017-03,374344.30,247303.02,51.37
7,2017-04,359927.23,374344.30,-3.85
8,2017-05,506071.14,359927.23,40.60
9,2017-06,433038.60,506071.14,-14.43


### Key Findings

- Month-over-Month (MoM) revenue growth was calculated using the `LAG()` window function.
- A continuous monthly timeline was created to ensure that each month is compared with the immediately preceding calendar month.
- Revenue showed substantial growth throughout much of 2017 and reached a major peak in November 2017.
- Product revenue increased from approximately 664,219 in October 2017 to 1,010,271 in November 2017, representing a 52.10% increase.
- Revenue declined by 26.36% in December 2017 following the November peak.
- During 2018, revenue remained relatively stable at a higher level, reaching approximately 1 million between March and May.
- Extremely large growth percentages during the early months are influenced by very small starting revenue values and should not be interpreted as normal business growth.
- The final months of the dataset contain very limited activity and may represent incomplete data periods; therefore, they should not be used to draw conclusions about business performance.

### Business Insight

The analysis indicates strong business growth over the observed period, with a significant revenue spike in November 2017 and a higher, relatively stable revenue level during early 2018.

In [7]:
for table in ['products', 'category_translation']:
    print(f"\n--- {table.upper()} ---")
    
    schema = pd.read_sql_query(
        f"PRAGMA table_info({table});",
        conn
    )
    
    print(schema[['name', 'type']].to_string(index=False))


--- PRODUCTS ---
                      name type
                product_id TEXT
     product_category_name TEXT
       product_name_lenght REAL
product_description_lenght REAL
        product_photos_qty REAL
          product_weight_g REAL
         product_length_cm REAL
         product_height_cm REAL
          product_width_cm REAL

--- CATEGORY_TRANSLATION ---
                         name type
        product_category_name TEXT
product_category_name_english TEXT


In [10]:
query = """
WITH category_revenue AS (

    SELECT
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        ) AS product_category,

        ROUND(SUM(oi.price), 2) AS product_revenue

    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    LEFT JOIN category_translation ct
        ON p.product_category_name = ct.product_category_name

    WHERE p.product_category_name IS NOT NULL

    GROUP BY
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        )
),

revenue_analysis AS (

    SELECT
        product_category,
        product_revenue,

        SUM(product_revenue) OVER () AS total_revenue,

        ROUND(
            (product_revenue * 100.0) /
            SUM(product_revenue) OVER (),
            2
        ) AS revenue_percentage,

        ROUND(
            SUM(product_revenue) OVER (
                ORDER BY product_revenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ),
            2
        ) AS cumulative_revenue

    FROM category_revenue
)

SELECT
    product_category,
    product_revenue,
    revenue_percentage,

    ROUND(
        (cumulative_revenue * 100.0) /
        total_revenue,
        2
    ) AS cumulative_revenue_percentage

FROM revenue_analysis

ORDER BY product_revenue DESC;
"""

pareto_analysis = pd.read_sql_query(query, conn)

pareto_analysis

,product_category,product_revenue,revenue_percentage,cumulative_revenue_percentage
0,health_beauty,1258681.34,9.38,9.38
1,watches_gifts,1205005.68,8.98,18.37
2,bed_bath_table,1036988.68,7.73,26.10
3,sports_leisure,988048.97,7.37,33.47
4,computers_accessories,911954.32,6.80,40.27
...,...,...,...,...
68,flowers,1110.04,0.01,99.98
69,home_comfort_2,760.27,0.01,99.99
70,cds_dvds_musicals,730.00,0.01,99.99
71,fashion_childrens_clothes,569.85,0.00,100.00


In [11]:
query = """
WITH category_revenue AS (

    SELECT
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        ) AS product_category,

        SUM(oi.price) AS product_revenue

    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    LEFT JOIN category_translation ct
        ON p.product_category_name = ct.product_category_name

    WHERE p.product_category_name IS NOT NULL

    GROUP BY
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        )
),

revenue_running_total AS (

    SELECT
        product_category,
        product_revenue,

        SUM(product_revenue) OVER () AS total_revenue,

        SUM(product_revenue) OVER (
            ORDER BY product_revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS cumulative_revenue

    FROM category_revenue
),

pareto AS (

    SELECT
        product_category,
        product_revenue,

        (cumulative_revenue * 100.0 / total_revenue)
            AS cumulative_revenue_percentage

    FROM revenue_running_total
)

SELECT
    COUNT(*) AS categories_required_for_80_percent_revenue

FROM pareto

WHERE cumulative_revenue_percentage < 80;
"""

pd.read_sql_query(query, conn)

,categories_required_for_80_percent_revenue
0,16


In [12]:
query = """
WITH category_revenue AS (

    SELECT
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        ) AS product_category,

        SUM(oi.price) AS product_revenue

    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    LEFT JOIN category_translation ct
        ON p.product_category_name = ct.product_category_name

    WHERE p.product_category_name IS NOT NULL

    GROUP BY
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        )
),

pareto AS (

    SELECT
        product_category,
        product_revenue,

        SUM(product_revenue) OVER () AS total_revenue,

        SUM(product_revenue) OVER (
            ORDER BY product_revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) * 100.0 /
        SUM(product_revenue) OVER () AS cumulative_revenue_percentage

    FROM category_revenue
)

SELECT
    product_category,
    ROUND(product_revenue, 2) AS product_revenue,
    ROUND(cumulative_revenue_percentage, 2)
        AS cumulative_revenue_percentage

FROM pareto

WHERE cumulative_revenue_percentage >= 75
  AND cumulative_revenue_percentage <= 85

ORDER BY cumulative_revenue_percentage;
"""

pd.read_sql_query(query, conn)

,product_category,product_revenue,cumulative_revenue_percentage
0,telephony,323667.53,75.26
1,office_furniture,273960.70,77.31
2,stationery,230943.23,79.03
3,computers,222963.13,80.69
4,pet_shop,214315.41,82.29
5,musical_instruments,191498.88,83.72


## 4.2 Revenue Concentration and Pareto Analysis

### Objective

To analyze how product revenue is distributed across product categories and determine whether a relatively small number of categories contribute a large proportion of total revenue.

### Data Quality Check

Before performing the analysis, category data was validated.

- There were **73 distinct named product categories**.
- **71 categories** had English translations available.
- **2 valid categories** did not have English translations.
- **1,603 order items** were associated with products that had missing category information.

To preserve valid categories while handling missing translations, `COALESCE()` was used to display the English category name when available and the original category name otherwise.

Products with missing category values were excluded only from this category-level analysis because they could not be assigned to a meaningful product category.

> No records in the original dataset were modified or deleted.

### Methodology

- Product revenue was calculated using `SUM(order_items.price)`.
- Products with known category information were included.
- Categories were ranked from highest to lowest revenue.
- Window functions were used to calculate cumulative revenue percentages.

### Key Findings

- The analysis included **73 valid product categories**.
- The highest-revenue category was **health_beauty**, contributing approximately **9.38%** of categorized product revenue.
- The top five categories contributed approximately **40.27%** of categorized product revenue.
- The top 16 categories accounted for approximately **79.03%** of categorized revenue.
- The top 17 categories accounted for approximately **80.69%** of categorized revenue.
- Therefore, approximately **23.3% of product categories generated over 80% of categorized product revenue**.

### Business Insight

Revenue is concentrated among a relatively small group of product categories. Although the results do not represent an exact 80/20 Pareto relationship, approximately **23% of product categories generate over 80% of categorized revenue**.

This indicates that the business is relatively dependent on its highest-performing product categories. These categories may be strategically important for inventory planning, marketing decisions, and revenue growth.

In [13]:
query = """
WITH dataset_max_date AS (
    
    SELECT
        MAX(DATE(order_purchase_timestamp)) AS max_purchase_date
    FROM orders
    WHERE order_status = 'delivered'
),

customer_rfm AS (

    SELECT
        c.customer_unique_id,

        MAX(DATE(o.order_purchase_timestamp)) AS last_purchase_date,

        CAST(
            julianday(
                (SELECT max_purchase_date FROM dataset_max_date)
            ) -
            julianday(MAX(DATE(o.order_purchase_timestamp)))
            AS INTEGER
        ) AS recency_days,

        COUNT(DISTINCT o.order_id) AS frequency,

        ROUND(SUM(oi.price), 2) AS monetary_value

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    customer_unique_id,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value

FROM customer_rfm

ORDER BY monetary_value DESC

LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_unique_id,last_purchase_date,recency_days,frequency,monetary_value
0,0a0a92112bd4c708ca5fde585afaa872,2017-09-29,334,1,13440.0
1,da122df9eeddfedc1dc1f5349a1a690c,2017-04-01,515,2,7388.0
2,763c8b1c9c68a0229c42c9fc6f662b93,2018-07-15,45,1,7160.0
3,dc4802a71eae9be1dd28f5d788ceb526,2017-02-12,563,1,6735.0
4,459bef486812aa25204be022145caa62,2018-07-25,35,1,6729.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,2017-05-24,462,1,6499.0
6,4007669dec559734d6f53e029e360987,2017-11-24,278,1,5934.6
7,eebb5dda148d3893cdaf5b5ca3040ccb,2017-04-18,498,1,4690.0
8,48e1ac109decbb87765a3eade6854098,2018-06-22,68,1,4590.0
9,a229eba70ec1c2abef51f04987deb7a5,2018-05-31,90,1,4400.0


In [14]:
query = """
WITH customer_rfm AS (

    SELECT
        c.customer_unique_id,

        COUNT(DISTINCT o.order_id) AS frequency

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    frequency,
    COUNT(*) AS total_customers,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS customer_percentage

FROM customer_rfm

GROUP BY frequency

ORDER BY frequency;
"""

pd.read_sql_query(query, conn)

,frequency,total_customers,customer_percentage
0,1,90557,97.00
1,2,2573,2.76
2,3,181,0.19
3,4,28,0.03
4,5,9,0.01
5,6,5,0.01
6,7,3,0.00
7,9,1,0.00
8,15,1,0.00


In [16]:
query = """
WITH dataset_max_date AS (

    SELECT
        MAX(DATE(order_purchase_timestamp)) AS max_purchase_date
    FROM orders
    WHERE order_status = 'delivered'
),

customer_metrics AS (

    SELECT
        c.customer_unique_id,

        MAX(DATE(o.order_purchase_timestamp)) AS last_purchase_date,

        CAST(
            julianday(
                (SELECT max_purchase_date FROM dataset_max_date)
            ) -
            julianday(MAX(DATE(o.order_purchase_timestamp)))
            AS INTEGER
        ) AS recency_days,

        COUNT(DISTINCT o.order_id) AS frequency,

        SUM(oi.price) AS monetary_value

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
),

scored_customers AS (

    SELECT
        customer_unique_id,
        last_purchase_date,
        recency_days,
        frequency,
        ROUND(monetary_value, 2) AS monetary_value,

        5 - NTILE(4) OVER (
            ORDER BY recency_days ASC
        ) AS recency_score,

        NTILE(4) OVER (
            ORDER BY monetary_value ASC
        ) AS monetary_score

    FROM customer_metrics
)

SELECT
    customer_unique_id,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    recency_score,
    monetary_score

FROM scored_customers

ORDER BY monetary_value DESC

LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_unique_id,last_purchase_date,recency_days,frequency,monetary_value,recency_score,monetary_score
0,0a0a92112bd4c708ca5fde585afaa872,2017-09-29,334,1,13440.0,2,4
1,da122df9eeddfedc1dc1f5349a1a690c,2017-04-01,515,2,7388.0,1,4
2,763c8b1c9c68a0229c42c9fc6f662b93,2018-07-15,45,1,7160.0,4,4
3,dc4802a71eae9be1dd28f5d788ceb526,2017-02-12,563,1,6735.0,1,4
4,459bef486812aa25204be022145caa62,2018-07-25,35,1,6729.0,4,4
5,ff4159b92c40ebe40454e3e6a7c35ed6,2017-05-24,462,1,6499.0,1,4
6,4007669dec559734d6f53e029e360987,2017-11-24,278,1,5934.6,2,4
7,eebb5dda148d3893cdaf5b5ca3040ccb,2017-04-18,498,1,4690.0,1,4
8,48e1ac109decbb87765a3eade6854098,2018-06-22,68,1,4590.0,4,4
9,a229eba70ec1c2abef51f04987deb7a5,2018-05-31,90,1,4400.0,4,4


In [17]:
query = """
WITH dataset_max_date AS (

    SELECT
        MAX(DATE(order_purchase_timestamp)) AS max_purchase_date
    FROM orders
    WHERE order_status = 'delivered'
),

customer_metrics AS (

    SELECT
        c.customer_unique_id,

        MAX(DATE(o.order_purchase_timestamp)) AS last_purchase_date,

        CAST(
            julianday(
                (SELECT max_purchase_date FROM dataset_max_date)
            ) -
            julianday(MAX(DATE(o.order_purchase_timestamp)))
            AS INTEGER
        ) AS recency_days,

        COUNT(DISTINCT o.order_id) AS frequency,

        SUM(oi.price) AS monetary_value

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
),

scored_customers AS (

    SELECT
        *,

        5 - NTILE(4) OVER (
            ORDER BY recency_days ASC
        ) AS recency_score,

        NTILE(4) OVER (
            ORDER BY monetary_value ASC
        ) AS monetary_score

    FROM customer_metrics
),

segmented_customers AS (

    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        ROUND(monetary_value, 2) AS monetary_value,
        recency_score,
        monetary_score,

        CASE
            WHEN recency_score >= 3
                 AND monetary_score >= 3
                THEN 'Recent High-Value'

            WHEN recency_score >= 3
                 AND monetary_score <= 2
                THEN 'Recent Low-Value'

            WHEN recency_score <= 2
                 AND monetary_score >= 3
                THEN 'Inactive High-Value'

            ELSE 'Inactive Low-Value'
        END AS customer_segment,

        CASE
            WHEN frequency > 1
                THEN 'Repeat Customer'
            ELSE 'One-Time Customer'
        END AS repeat_behavior

    FROM scored_customers
)

SELECT
    customer_segment,
    repeat_behavior,

    COUNT(*) AS total_customers,

    ROUND(
        AVG(recency_days),
        2
    ) AS average_recency_days,

    ROUND(
        AVG(monetary_value),
        2
    ) AS average_customer_value,

    ROUND(
        SUM(monetary_value),
        2
    ) AS total_revenue

FROM segmented_customers

GROUP BY
    customer_segment,
    repeat_behavior

ORDER BY
    total_revenue DESC;
"""

pd.read_sql_query(query, conn)

,customer_segment,repeat_behavior,total_customers,average_recency_days,average_customer_value,total_revenue
0,Recent High-Value,One-Time Customer,22289,111.54,231.63,5162845.42
1,Inactive High-Value,One-Time Customer,22046,364.30,233.71,5152298.94
2,Inactive Low-Value,One-Time Customer,23374,363.68,47.22,1103695.61
3,Recent Low-Value,One-Time Customer,22848,111.02,47.02,1074249.39
4,Recent High-Value,Repeat Customer,1310,112.34,300.23,393299.50
5,Inactive High-Value,Repeat Customer,1033,347.46,296.90,306700.94
6,Recent Low-Value,Repeat Customer,233,110.46,62.17,14485.46
7,Inactive Low-Value,Repeat Customer,225,372.71,61.88,13922.85


In [20]:
query = """
WITH customer_rfm AS (
    
    SELECT
        c.customer_unique_id,
        
        MAX(DATE(o.order_purchase_timestamp)) AS last_purchase_date,
        
        CAST(
            JULIANDAY(
                (SELECT MAX(DATE(order_purchase_timestamp)) FROM orders)
            ) 
            - JULIANDAY(MAX(DATE(o.order_purchase_timestamp)))
            AS INTEGER
        ) AS recency_days,
        
        COUNT(DISTINCT o.order_id) AS frequency,
        
        ROUND(SUM(oi.price), 2) AS monetary_value

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
),

scored_customers AS (

    SELECT
        *,
        
        NTILE(4) OVER (
            ORDER BY recency_days DESC
        ) AS recency_score,
        
        NTILE(4) OVER (
            ORDER BY monetary_value
        ) AS monetary_score

    FROM customer_rfm
),

segmented_customers AS (

    SELECT
        *,
        
        CASE
            WHEN frequency > 1 THEN 'Repeat Customer'
            ELSE 'One-Time Customer'
        END AS customer_type,
        
        CASE
            WHEN recency_score >= 3 
                 AND monetary_score >= 3
                THEN 'More Recent - Higher Spending'
                
            WHEN recency_score >= 3 
                 AND monetary_score < 3
                THEN 'More Recent - Lower Spending'
                
            WHEN recency_score < 3 
                 AND monetary_score >= 3
                THEN 'Less Recent - Higher Spending'
                
            ELSE 'Less Recent - Lower Spending'
        END AS customer_segment

    FROM scored_customers
)

SELECT
    customer_segment,
    customer_type,
    
    COUNT(*) AS total_customers,
    
    ROUND(AVG(recency_days), 2) 
        AS average_recency_days,
    
    ROUND(AVG(monetary_value), 2) 
        AS average_customer_value,
    
    ROUND(SUM(monetary_value), 2) 
        AS total_revenue

FROM segmented_customers

GROUP BY
    customer_segment,
    customer_type

ORDER BY
    total_revenue DESC;
"""

df_segments = pd.read_sql_query(query, conn)

df_segments

,customer_segment,customer_type,total_customers,average_recency_days,average_customer_value,total_revenue
0,More Recent - Higher Spending,One-Time Customer,22289,160.54,231.63,5162845.42
1,Less Recent - Higher Spending,One-Time Customer,22046,413.30,233.71,5152298.94
2,Less Recent - Lower Spending,One-Time Customer,23376,412.67,47.22,1103718.15
3,More Recent - Lower Spending,One-Time Customer,22846,160.01,47.02,1074226.85
4,More Recent - Higher Spending,Repeat Customer,1310,161.34,300.23,393299.50
5,Less Recent - Higher Spending,Repeat Customer,1033,396.46,296.90,306700.94
6,More Recent - Lower Spending,Repeat Customer,233,159.46,62.17,14485.46
7,Less Recent - Lower Spending,Repeat Customer,225,421.71,61.88,13922.85


## 4.3 Customer Segmentation (Adapted RFM)

This analysis segments customers using an adapted RFM approach.

Traditional RFM analysis evaluates customers based on:

- **Recency:** How recently a customer made a purchase.
- **Frequency:** How often a customer made purchases.
- **Monetary Value:** The total product value generated by a customer.

### Adaptation for this Dataset

The purchase frequency distribution in this dataset is highly concentrated among one-time customers. Therefore, Frequency is used separately to classify customers as:

- **One-Time Customer**
- **Repeat Customer**

Customer segments are primarily determined using Recency and Monetary Value:

- **More Recent - Higher Spending**
- **More Recent - Lower Spending**
- **Less Recent - Higher Spending**
- **Less Recent - Lower Spending**

This approach provides a clearer view of customer value and purchasing behavior while accounting for the dataset's highly skewed frequency distribution.

> Note: This segmentation is descriptive and does not imply causation or predict future customer behavior.

In [21]:
query = """
WITH delivery_data AS (

    SELECT
        o.order_id,

        CAST(
            JULIANDAY(o.order_delivered_customer_date) -
            JULIANDAY(o.order_estimated_delivery_date)
            AS REAL
        ) AS delivery_delay_days,

        r.review_score

    FROM orders o

    LEFT JOIN reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

delivery_segments AS (

    SELECT
        *,
        
        CASE
            WHEN delivery_delay_days <= 0
                THEN 'On Time or Early'

            WHEN delivery_delay_days <= 7
                THEN '1-7 Days Late'

            WHEN delivery_delay_days <= 14
                THEN '8-14 Days Late'

            ELSE '15+ Days Late'

        END AS delay_severity

    FROM delivery_data
)

SELECT
    delay_severity,

    COUNT(DISTINCT order_id) AS total_orders,

    COUNT(review_score) AS total_reviews,

    ROUND(AVG(review_score), 2) AS average_review_score

FROM delivery_segments

GROUP BY delay_severity

ORDER BY
    CASE delay_severity
        WHEN 'On Time or Early' THEN 1
        WHEN '1-7 Days Late' THEN 2
        WHEN '8-14 Days Late' THEN 3
        WHEN '15+ Days Late' THEN 4
    END;
"""

df_delay_severity = pd.read_sql_query(query, conn)

df_delay_severity

,delay_severity,total_orders,total_reviews,average_review_score
0,On Time or Early,88644,88653,4.29
1,1-7 Days Late,4481,4428,3.18
2,8-14 Days Late,1790,1758,1.75
3,15+ Days Late,1555,1514,1.71


,late_delivery_rate_percentage,average_review_score
count,425.000000,425.000000
mean,7.835129,4.140800
std,4.860574,0.291922
min,0.000000,2.390000
25%,4.410000,3.990000
50%,6.780000,4.170000
75%,10.480000,4.330000
max,30.140000,4.820000


In [35]:
query = """
WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_review_score
    FROM reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),

delivery_analysis AS (
    SELECT
        o.order_id,

        CAST(
            julianday(date(o.order_delivered_customer_date))
            - julianday(date(o.order_estimated_delivery_date))
            AS INTEGER
        ) AS delivery_delay_days,

        r.order_review_score

    FROM orders o

    LEFT JOIN order_reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)

SELECT
    CASE
        WHEN delivery_delay_days <= 0
            THEN 'On Time or Early'

        WHEN delivery_delay_days BETWEEN 1 AND 7
            THEN '1-7 Days Late'

        WHEN delivery_delay_days BETWEEN 8 AND 14
            THEN '8-14 Days Late'

        ELSE '15+ Days Late'
    END AS delay_severity,

    COUNT(*) AS total_orders,

    COUNT(order_review_score) AS orders_with_reviews,

    ROUND(AVG(order_review_score), 2) AS average_review_score

FROM delivery_analysis

GROUP BY delay_severity

ORDER BY
    CASE delay_severity
        WHEN 'On Time or Early' THEN 1
        WHEN '1-7 Days Late' THEN 2
        WHEN '8-14 Days Late' THEN 3
        WHEN '15+ Days Late' THEN 4
    END;
"""

df_delay_reviews = pd.read_sql_query(query, conn)

df_delay_reviews

,delay_severity,total_orders,orders_with_reviews,average_review_score
0,On Time or Early,89936,89443,4.29
1,1-7 Days Late,3672,3600,2.72
2,8-14 Days Late,1478,1446,1.67
3,15+ Days Late,1384,1335,1.73


## 4.4 Delivery Delay Severity vs Customer Reviews

This analysis examines the relationship between delivery timeliness and customer review scores.

Delivery delay is calculated by comparing the actual customer delivery date with the estimated delivery date.

To avoid multiple review records disproportionately influencing the results, review scores are first aggregated to one average review score per order.

Orders are then grouped based on delivery delay severity:

- **On Time or Early:** Delivered on or before the estimated delivery date.
- **1–7 Days Late:** Delivered 1 to 7 days after the estimated delivery date.
- **8–14 Days Late:** Delivered 8 to 14 days after the estimated delivery date.
- **15+ Days Late:** Delivered 15 or more days after the estimated delivery date.

### Important Methodological Note

Delivery and review outcomes are analyzed at the order level.

Some orders may contain multiple review records. Therefore, review scores are aggregated to one average review score per order before calculating the average review score for each delivery category.

The analysis identifies an association between delivery delay severity and customer review scores and does not establish causation.

In [36]:
query = """
WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_review_score
    FROM reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),

order_delivery AS (
    SELECT
        o.order_id,

        CASE
            WHEN date(o.order_delivered_customer_date)
                 > date(o.order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END AS is_late,

        r.order_review_score

    FROM orders o

    LEFT JOIN order_reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

seller_orders AS (
    SELECT
        oi.seller_id,
        oi.order_id,
        od.is_late,
        od.order_review_score,

        SUM(oi.price) AS order_product_revenue,
        COUNT(*) AS items_in_order

    FROM order_items oi

    INNER JOIN order_delivery od
        ON oi.order_id = od.order_id

    GROUP BY
        oi.seller_id,
        oi.order_id,
        od.is_late,
        od.order_review_score
),

seller_performance AS (
    SELECT
        so.seller_id,
        s.seller_city,
        s.seller_state,

        COUNT(DISTINCT so.order_id) AS total_delivered_orders,

        SUM(so.items_in_order) AS total_items_sold,

        ROUND(SUM(so.order_product_revenue), 2) AS product_revenue,

        ROUND(
            100.0 * AVG(so.is_late),
            2
        ) AS late_delivery_rate_percentage,

        ROUND(
            AVG(so.order_review_score),
            2
        ) AS average_review_score

    FROM seller_orders so

    LEFT JOIN sellers s
        ON so.seller_id = s.seller_id

    GROUP BY
        so.seller_id,
        s.seller_city,
        s.seller_state

    HAVING COUNT(DISTINCT so.order_id) >= 50
)

SELECT *
FROM seller_performance

ORDER BY late_delivery_rate_percentage DESC;
"""

df_seller_performance = pd.read_sql_query(query, conn)

df_seller_performance

,seller_id,seller_city,seller_state,total_delivered_orders,total_items_sold,product_revenue,late_delivery_rate_percentage,average_review_score
0,54965bbe3e4f07ae045b90b0b8541f52,foz do iguacu,PR,73,81,10351.70,30.14,3.14
1,beadbee30901a7f61d031b6b686095ad,guarulhos,SP,64,68,4373.98,23.44,3.91
2,a49928bcdf77c55c6d6e05e09a9b4ca5,sao paulo,SP,96,104,8646.90,21.88,3.03
3,712e6ed8aa4aa1fa65dab41fed5737e4,videira,SC,77,85,39385.00,20.78,3.39
4,06a2c3af7b3aee5d69171b0e14f0ee87,sao luis,MA,389,402,36097.98,19.02,4.01
...,...,...,...,...,...,...,...,...
420,d57e18d5f73c7ccb7f7339b61166898d,sao paulo,SP,61,63,3187.70,0.00,4.43
421,e882b2a25a10b9c057cc49695f222c19,teresopolis,RJ,57,58,51057.54,0.00,4.58
422,f3b80352b986ab4d1057a4b724be19d0,brasilia,DF,86,91,9815.10,0.00,4.21
423,fa1a9dec3a9940c072684a46728bf1fc,icara,SC,55,60,6674.00,0.00,4.11


In [38]:
# Calculate dataset-relative benchmarks

late_delivery_threshold = (
    df_seller_performance['late_delivery_rate_percentage']
    .quantile(0.75)
)

review_score_threshold = (
    df_seller_performance['average_review_score']
    .quantile(0.25)
)


# Create a copy of the seller performance dataframe

df_seller_risk = df_seller_performance.copy()


# Classify seller-associated performance using dataset-relative benchmarks

def classify_seller(row):

    if (
        row['late_delivery_rate_percentage'] > late_delivery_threshold
        and row['average_review_score'] < review_score_threshold
    ):
        return 'Combined Delivery & Satisfaction Concern'

    elif row['late_delivery_rate_percentage'] > late_delivery_threshold:
        return 'Higher Delivery Concern'

    elif row['average_review_score'] < review_score_threshold:
        return 'Lower Satisfaction Concern'

    else:
        return 'No Relative Concern Identified'


df_seller_risk['seller_performance_category'] = (
    df_seller_risk.apply(classify_seller, axis=1)
)


df_seller_risk[
    [
        'seller_id',
        'seller_city',
        'seller_state',
        'total_delivered_orders',
        'late_delivery_rate_percentage',
        'average_review_score',
        'seller_performance_category'
    ]
].head()

,seller_id,seller_city,seller_state,total_delivered_orders,late_delivery_rate_percentage,average_review_score,seller_performance_category
0,54965bbe3e4f07ae045b90b0b8541f52,foz do iguacu,PR,73,30.14,3.14,Combined Delivery & Satisfaction Concern
1,beadbee30901a7f61d031b6b686095ad,guarulhos,SP,64,23.44,3.91,Combined Delivery & Satisfaction Concern
2,a49928bcdf77c55c6d6e05e09a9b4ca5,sao paulo,SP,96,21.88,3.03,Combined Delivery & Satisfaction Concern
3,712e6ed8aa4aa1fa65dab41fed5737e4,videira,SC,77,20.78,3.39,Combined Delivery & Satisfaction Concern
4,06a2c3af7b3aee5d69171b0e14f0ee87,sao luis,MA,389,19.02,4.01,Higher Delivery Concern


In [39]:
seller_risk_summary = (
    df_seller_risk
    .groupby('seller_performance_category')
    .agg(
        total_sellers=('seller_id', 'count'),
        average_late_delivery_rate=(
            'late_delivery_rate_percentage',
            'mean'
        ),
        average_review_score=(
            'average_review_score',
            'mean'
        )
    )
    .reset_index()
)


# Round percentage and average values

seller_risk_summary['average_late_delivery_rate'] = (
    seller_risk_summary['average_late_delivery_rate']
    .round(2)
)

seller_risk_summary['average_review_score'] = (
    seller_risk_summary['average_review_score']
    .round(2)
)


# Calculate percentage of sellers in each category

seller_risk_summary['seller_percentage'] = (
    seller_risk_summary['total_sellers']
    / seller_risk_summary['total_sellers'].sum()
    * 100
).round(2)


# Display the summary

seller_risk_summary

,seller_performance_category,total_sellers,average_late_delivery_rate,average_review_score,seller_percentage
0,Combined Delivery & Satisfaction Concern,52,13.66,3.69,12.24
1,Higher Delivery Concern,54,11.26,4.18,12.71
2,Lower Satisfaction Concern,53,5.23,3.84,12.47
3,No Relative Concern Identified,266,4.41,4.28,62.59


## 4.5 Seller-Associated Operational Performance and Relative Risk Indicators

This analysis extends the previous seller revenue analysis by examining operational and customer satisfaction indicators associated with seller orders.

For each seller, the analysis evaluates:

- Total delivered orders
- Total items sold
- Product revenue
- Late delivery rate
- Average customer review score

To ensure more reliable comparisons, the analysis includes only sellers with at least 50 delivered orders.

### Important Methodological Note

Delivery and review information is recorded at the order level. An order may contain products from multiple sellers.

Therefore, the results represent **seller-associated order performance** and do not establish that a specific seller directly caused a delivery delay or customer review outcome.

### Dataset-Relative Performance Benchmarks

Seller performance indicators are evaluated relative to the distribution of the analyzed sellers.

The following dataset-relative benchmarks are used:

- Sellers above the 75th percentile of late delivery rates are identified as having a higher delivery concern.
- Sellers below the 25th percentile of average review scores are identified as having a lower satisfaction concern.

Based on these indicators, sellers are classified into:

- **Combined Delivery & Satisfaction Concern:** Higher late-delivery rates and lower review scores.
- **Higher Delivery Concern:** Higher late-delivery rates only.
- **Lower Satisfaction Concern:** Lower review scores only.
- **No Relative Concern Identified:** Does not fall into either relative concern category.

These classifications are relative to the analyzed seller population and should not be interpreted as universal business performance ratings.

In [29]:
query = """
SELECT
    COUNT(*) AS total_review_records,
    COUNT(DISTINCT order_id) AS orders_with_reviews,
    COUNT(*) - COUNT(DISTINCT order_id) AS extra_review_records
FROM reviews;
"""

df_review_validation = pd.read_sql_query(query, conn)

df_review_validation

,total_review_records,orders_with_reviews,extra_review_records
0,99224,98673,551


In [30]:
query = """
SELECT
    order_id,
    COUNT(*) AS review_records
FROM reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY review_records DESC
LIMIT 10;
"""

df_multiple_reviews = pd.read_sql_query(query, conn)

df_multiple_reviews

,order_id,review_records
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2
5,ff850ba359507b996e8b2fbb26df8d03,2
6,ff763b73e473d03c321bcd5a053316e8,2
7,fe041ba1c9f54016432fa6ee91709dbc,2
8,fd95ae805c63c534f1a64589e102225e,2
9,fd61441ba2a7b57e6342862e779b10b0,2


In [11]:
query = """
WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_review_score
    FROM reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),

delivery_analysis AS (
    SELECT
        o.order_id,
        r.order_review_score,

        CAST(
            julianday(date(o.order_delivered_customer_date))
            - julianday(date(o.order_estimated_delivery_date))
            AS INTEGER
        ) AS delivery_delay_days

    FROM orders o

    LEFT JOIN order_reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)

SELECT
    CASE
        WHEN delivery_delay_days <= 0
            THEN 'On Time or Early'

        WHEN delivery_delay_days BETWEEN 1 AND 7
            THEN '1-7 Days Late'

        WHEN delivery_delay_days BETWEEN 8 AND 14
            THEN '8-14 Days Late'

        ELSE '15+ Days Late'
    END AS delay_severity,

    COUNT(*) AS total_orders,

    COUNT(order_review_score) AS orders_with_reviews,

    ROUND(AVG(order_review_score), 2) AS average_review_score

FROM delivery_analysis

GROUP BY delay_severity

ORDER BY
    CASE delay_severity
        WHEN 'On Time or Early' THEN 1
        WHEN '1-7 Days Late' THEN 2
        WHEN '8-14 Days Late' THEN 3
        WHEN '15+ Days Late' THEN 4
    END;
"""

df_delay_reviews = pd.read_sql_query(query, conn)

df_delay_reviews

,delay_severity,total_orders,orders_with_reviews,average_review_score
0,On Time or Early,89936,89443,4.29
1,1-7 Days Late,3672,3600,2.72
2,8-14 Days Late,1478,1446,1.67
3,15+ Days Late,1384,1335,1.73


# 4.X Product Category and Customer Satisfaction Analysis

This analysis examines whether customer satisfaction varies across product categories.

For each product category, the analysis evaluates:

- Total delivered orders
- Average review score
- Percentage of low-rated reviews (1–2 stars)
- Product revenue

This helps identify categories that may require further investigation, particularly categories with substantial business volume and relatively lower customer satisfaction.

The analysis identifies associations and does not establish that product category directly causes differences in customer satisfaction.

In [42]:
query = """

WITH order_reviews AS (

    SELECT
        order_id,
        AVG(review_score) AS order_review_score

    FROM reviews

    WHERE review_score IS NOT NULL

    GROUP BY order_id

),

order_category_data AS (

    SELECT DISTINCT
        o.order_id,
        p.product_category_name AS product_category

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE o.order_status = 'delivered'
      AND p.product_category_name IS NOT NULL

),

category_revenue AS (

    SELECT
        p.product_category_name AS product_category,

        ROUND(SUM(oi.price), 2) AS product_revenue

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE o.order_status = 'delivered'
      AND p.product_category_name IS NOT NULL

    GROUP BY p.product_category_name

)

SELECT

    ocd.product_category,

    COUNT(DISTINCT ocd.order_id) AS total_delivered_orders,

    COUNT(r.order_review_score) AS orders_with_reviews,

    ROUND(AVG(r.order_review_score), 2) AS average_review_score,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN r.order_review_score IN (1, 2)
                THEN 1
                ELSE 0
            END
        )
        / COUNT(r.order_review_score),
        2
    ) AS low_rating_percentage,

    cr.product_revenue

FROM order_category_data ocd

LEFT JOIN order_reviews r
    ON ocd.order_id = r.order_id

JOIN category_revenue cr
    ON ocd.product_category = cr.product_category

GROUP BY
    ocd.product_category,
    cr.product_revenue

HAVING COUNT(DISTINCT ocd.order_id) >= 50

ORDER BY average_review_score ASC;

"""

df_category_satisfaction = pd.read_sql_query(query, conn)

df_category_satisfaction

,product_category,total_delivered_orders,orders_with_reviews,average_review_score,low_rating_percentage,product_revenue
0,moveis_escritorio,1254,1244,3.64,21.95,268154.31
1,fashion_roupa_masculina,106,105,3.82,22.86,10452.33
2,audio,348,345,3.84,21.45,50570.60
3,casa_conforto,392,390,3.89,18.72,58008.45
4,construcao_ferramentas_seguranca,159,158,3.97,17.09,38773.22
5,telefonia_fixa,212,209,3.97,16.75,55315.21
6,casa_construcao,483,481,3.99,17.26,81407.57
7,cama_mesa_banho,9272,9177,4.00,15.88,1023434.76
8,fashion_underwear_e_moda_praia,117,116,4.01,16.38,9305.95
9,telefonia,4093,4069,4.05,14.13,309860.23


### Key Findings

Product categories showed variation in customer review scores, indicating that customer satisfaction was not uniform across the product portfolio.

Among categories with at least 50 delivered orders, `moveis_escritorio` (office furniture) recorded the lowest average review score at 3.64, with 21.95% of reviewed orders receiving a low rating of 1 or 2 stars. This category also represented a substantial volume of 1,254 delivered orders.

Some other categories, including fashion clothing and audio products, also showed relatively low review scores and higher proportions of low ratings. However, these categories had lower order volumes and should therefore be interpreted more cautiously.

Business impact should consider both customer satisfaction and transaction volume. For example, `cama_mesa_banho` (bed, bath and table) recorded 9,272 delivered orders and approximately 1.02 million in product revenue, while having an average review score of 4.00 and a low-rating percentage of 15.88%.

These results suggest that product categories may be associated with different customer experience outcomes. However, order-level review scores can reflect multiple aspects of the purchasing experience, including delivery performance and seller-associated factors. Therefore, the analysis does not establish that the product category itself caused differences in customer satisfaction.

# 4.X Geographic Delivery Performance and Customer Satisfaction Analysis

This analysis examines whether customer experience varies across geographic regions.

For each customer state, the analysis evaluates:

- Total delivered orders
- Average delivery time
- Late delivery rate
- Average customer review score

This combined analysis helps identify geographic patterns where delivery performance and customer satisfaction may require further investigation.

The results identify associations between customer location, delivery performance, and review scores. They do not establish that geographic location directly causes differences in customer satisfaction.

In [43]:
query = """

WITH order_reviews AS (

    SELECT
        order_id,
        AVG(review_score) AS order_review_score

    FROM reviews

    WHERE review_score IS NOT NULL

    GROUP BY order_id

),

state_delivery_analysis AS (

    SELECT

        c.customer_state,

        o.order_id,

        CAST(
            julianday(date(o.order_delivered_customer_date))
            -
            julianday(date(o.order_purchase_timestamp))
            AS INTEGER
        ) AS delivery_time_days,

        CASE
            WHEN date(o.order_delivered_customer_date)
                 > date(o.order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END AS late_delivery,

        r.order_review_score

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    LEFT JOIN order_reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'

      AND o.order_delivered_customer_date IS NOT NULL

      AND o.order_purchase_timestamp IS NOT NULL

      AND o.order_estimated_delivery_date IS NOT NULL

)

SELECT

    customer_state,

    COUNT(DISTINCT order_id) AS total_delivered_orders,

    ROUND(
        AVG(delivery_time_days),
        2
    ) AS average_delivery_time_days,

    ROUND(
        100.0 * AVG(late_delivery),
        2
    ) AS late_delivery_rate_percentage,

    COUNT(order_review_score) AS orders_with_reviews,

    ROUND(
        AVG(order_review_score),
        2
    ) AS average_review_score

FROM state_delivery_analysis

GROUP BY customer_state

HAVING COUNT(DISTINCT order_id) >= 100

ORDER BY average_review_score ASC;

"""

df_geographic_experience = pd.read_sql_query(query, conn)

df_geographic_experience

,customer_state,total_delivered_orders,average_delivery_time_days,late_delivery_rate_percentage,orders_with_reviews,average_review_score
0,MA,717,21.51,17.43,712,3.83
1,AL,397,24.50,21.41,394,3.85
2,PA,946,23.73,11.21,933,3.91
3,SE,335,21.46,15.22,334,3.91
4,BA,3256,19.28,12.16,3229,3.93
5,CE,1279,21.20,13.76,1273,3.94
6,RJ,12350,15.24,12.11,12211,3.97
7,PI,476,19.39,13.87,471,3.99
8,ES,1995,15.72,10.73,1969,4.08
9,PB,517,20.39,10.44,512,4.08


### Geographic Performance Classification

To provide a more structured interpretation of geographic patterns, customer states are compared against dataset-level benchmarks for late delivery rate and average review score.

States are classified based on whether they show:

- A higher-than-average late delivery rate
- A lower-than-average customer review score

The classification identifies relative areas of concern within the dataset and does not establish that customer location directly causes differences in delivery performance or customer satisfaction.

In [44]:
query = """

WITH order_reviews AS (

    SELECT
        order_id,
        AVG(review_score) AS order_review_score

    FROM reviews

    WHERE review_score IS NOT NULL

    GROUP BY order_id

),

state_metrics AS (

    SELECT

        c.customer_state,

        COUNT(DISTINCT o.order_id) AS total_delivered_orders,

        ROUND(
            AVG(
                julianday(date(o.order_delivered_customer_date))
                -
                julianday(date(o.order_purchase_timestamp))
            ),
            2
        ) AS average_delivery_time_days,

        ROUND(
            100.0 * AVG(
                CASE
                    WHEN date(o.order_delivered_customer_date)
                         > date(o.order_estimated_delivery_date)
                    THEN 1.0
                    ELSE 0.0
                END
            ),
            2
        ) AS late_delivery_rate_percentage,

        ROUND(
            AVG(r.order_review_score),
            2
        ) AS average_review_score

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    LEFT JOIN order_reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'

      AND o.order_delivered_customer_date IS NOT NULL

      AND o.order_purchase_timestamp IS NOT NULL

      AND o.order_estimated_delivery_date IS NOT NULL

    GROUP BY c.customer_state

    HAVING COUNT(DISTINCT o.order_id) >= 100

),

benchmarks AS (

    SELECT

        AVG(late_delivery_rate_percentage)
            AS avg_late_delivery_rate,

        AVG(average_review_score)
            AS avg_review_score

    FROM state_metrics

)

SELECT

    sm.customer_state,

    sm.total_delivered_orders,

    sm.average_delivery_time_days,

    sm.late_delivery_rate_percentage,

    sm.average_review_score,

    CASE

        WHEN sm.late_delivery_rate_percentage >
             b.avg_late_delivery_rate

         AND sm.average_review_score <
             b.avg_review_score

            THEN 'Combined Delivery & Satisfaction Concern'


        WHEN sm.late_delivery_rate_percentage >
             b.avg_late_delivery_rate

            THEN 'Higher Delivery Concern'


        WHEN sm.average_review_score <
             b.avg_review_score

            THEN 'Lower Satisfaction Concern'


        ELSE 'No Relative Concern Identified'

    END AS geographic_performance_category

FROM state_metrics sm

CROSS JOIN benchmarks b

ORDER BY

    CASE geographic_performance_category

        WHEN 'Combined Delivery & Satisfaction Concern' THEN 1

        WHEN 'Higher Delivery Concern' THEN 2

        WHEN 'Lower Satisfaction Concern' THEN 3

        WHEN 'No Relative Concern Identified' THEN 4

    END,

    sm.average_review_score ASC;

"""

df_geographic_classification = pd.read_sql_query(query, conn)

df_geographic_classification

,customer_state,total_delivered_orders,average_delivery_time_days,late_delivery_rate_percentage,average_review_score,geographic_performance_category
0,MA,717,21.51,17.43,3.83,Combined Delivery & Satisfaction Concern
1,AL,397,24.50,21.41,3.85,Combined Delivery & Satisfaction Concern
2,PA,946,23.73,11.21,3.91,Combined Delivery & Satisfaction Concern
3,SE,335,21.46,15.22,3.91,Combined Delivery & Satisfaction Concern
4,BA,3256,19.28,12.16,3.93,Combined Delivery & Satisfaction Concern
5,CE,1279,21.20,13.76,3.94,Combined Delivery & Satisfaction Concern
6,RJ,12350,15.24,12.11,3.97,Combined Delivery & Satisfaction Concern
7,PI,476,19.39,13.87,3.99,Combined Delivery & Satisfaction Concern
8,ES,1995,15.72,10.73,4.08,Higher Delivery Concern
9,PB,517,20.39,10.44,4.08,Higher Delivery Concern


### Key Findings

Customer experience varied across geographic regions, with differences observed in delivery performance and average review scores.

Several states showed a combined relative concern pattern, meaning that they had both a higher-than-benchmark late delivery rate and a lower-than-benchmark average review score among states included in the analysis. These states included MA, AL, PA, SE, BA, CE, RJ, and PI. The classification is based on relative dataset benchmarks and identifies areas for further investigation rather than objectively poor-performing states.

The results also indicate that delivery duration alone does not fully explain customer satisfaction. For example, AM recorded the longest average delivery time at 26.36 days but had a low late delivery rate of 2.76% and an average review score of 4.24.

In contrast, AL had a similarly long average delivery time of 24.50 days but a substantially higher late delivery rate of 21.41% and a lower average review score of 3.85.

This pattern suggests that meeting the expected delivery timeline may be more strongly associated with customer satisfaction than delivery duration alone. However, the analysis identifies associations and does not establish that delivery performance directly caused differences in customer review scores.

Geographic performance patterns should therefore be interpreted alongside order volume. States with relatively high transaction volumes and weaker delivery or satisfaction metrics may represent higher-priority areas for further operational investigation.

# Customer Experience and Repeat Purchase Analysis

## Objective

To investigate whether the experience associated with a customer's first observed delivered order is related to their likelihood of making another purchase.

Customers are grouped according to the review score of their first observed delivered order:

- Low-rated experience: 1–2 stars
- High-rated experience: 4–5 stars

The analysis uses a 90-day observation window for repeat purchases to reduce bias caused by customers entering the dataset near its end.

This analysis identifies associations and does not establish that review scores directly cause future purchasing behavior.

In [45]:
query = """
SELECT
    MIN(order_purchase_timestamp) AS first_order_date,
    MAX(order_purchase_timestamp) AS last_order_date,
    COUNT(*) AS total_orders
FROM orders;
"""

df_order_date_range = pd.read_sql_query(query, conn)

df_order_date_range

,first_order_date,last_order_date,total_orders
0,2016-09-04 21:15:19,2018-10-17 17:30:18,99441


In [6]:
query = """

WITH order_reviews AS (

    SELECT
        order_id,
        AVG(review_score) AS review_score
    FROM reviews
    GROUP BY order_id

),

customer_orders AS (

    SELECT
        c.customer_unique_id,
        o.order_id,
        DATE(o.order_purchase_timestamp) AS purchase_date,
        r.review_score,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp, o.order_id
        ) AS order_number,

        LEAD(
            DATE(o.order_purchase_timestamp)
        ) OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp, o.order_id
        ) AS next_purchase_date

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    LEFT JOIN order_reviews r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'

),

first_orders AS (

    SELECT
        customer_unique_id,
        order_id,
        purchase_date,
        review_score,
        next_purchase_date,

        CASE
            WHEN review_score IN (1, 2)
                THEN 'Low Rated (1-2)'

            WHEN review_score IN (4, 5)
                THEN 'High Rated (4-5)'

            ELSE NULL

        END AS experience_group

    FROM customer_orders

    WHERE order_number = 1

),

eligible_customers AS (

    SELECT
        *,

        CASE
            WHEN next_purchase_date IS NOT NULL
             AND next_purchase_date <= DATE(
                    purchase_date,
                    '+90 days'
                 )
            THEN 1

            ELSE 0

        END AS repeated_within_90_days

    FROM first_orders

    WHERE purchase_date <= '2018-07-19'

      AND experience_group IS NOT NULL

)

SELECT

    experience_group,

    COUNT(*) AS customers,

    SUM(repeated_within_90_days)
        AS repeat_customers,

    ROUND(
        100.0 *
        SUM(repeated_within_90_days)
        / COUNT(*),
        2
    ) AS repeat_purchase_rate_percentage

FROM eligible_customers

GROUP BY experience_group

ORDER BY experience_group;

"""

df_experience_repeat = pd.read_sql_query(query, conn)

df_experience_repeat

,experience_group,customers,repeat_customers,repeat_purchase_rate_percentage
0,High Rated (4-5),65718,1401,2.13
1,Low Rated (1-2),11038,232,2.10


In [7]:
pd.read_sql_query("SELECT 1 AS test", conn)

,test
0,1


In [8]:
query = """

WITH first_orders AS (

    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp
        ) AS order_rank

    FROM orders o

    JOIN customers c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'

),

customer_first_order AS (

    SELECT
        customer_unique_id,
        order_id AS first_order_id,
        order_purchase_timestamp AS first_order_date,
        order_delivered_customer_date,
        order_estimated_delivery_date,

        CASE
            WHEN order_delivered_customer_date
                 <= order_estimated_delivery_date
            THEN 'On Time / Early'

            WHEN order_delivered_customer_date
                 > order_estimated_delivery_date
            THEN 'Late'

        END AS first_delivery_experience

    FROM first_orders

    WHERE order_rank = 1

),

eligible_customers AS (

    SELECT *

    FROM customer_first_order

    WHERE first_order_date <=
        (
            SELECT DATE(
                MAX(order_purchase_timestamp),
                '-90 days'
            )
            FROM orders
        )

),

repeat_status AS (

    SELECT
        e.customer_unique_id,
        e.first_delivery_experience,

        CASE
            WHEN COUNT(o.order_id) > 0
            THEN 1

            ELSE 0

        END AS repeated_within_90_days

    FROM eligible_customers e

    LEFT JOIN customers c
        ON e.customer_unique_id = c.customer_unique_id

    LEFT JOIN orders o
        ON c.customer_id = o.customer_id
        AND o.order_purchase_timestamp > e.first_order_date
        AND o.order_purchase_timestamp <=
            DATE(e.first_order_date, '+90 days')

    GROUP BY
        e.customer_unique_id,
        e.first_delivery_experience

)

SELECT

    first_delivery_experience,

    COUNT(*) AS customers,

    SUM(repeated_within_90_days)
        AS repeat_customers,

    ROUND(
        100.0 *
        SUM(repeated_within_90_days)
        / COUNT(*),
        2
    ) AS repeat_purchase_rate_percentage

FROM repeat_status

WHERE first_delivery_experience IS NOT NULL

GROUP BY first_delivery_experience

ORDER BY first_delivery_experience;

"""

df_first_delivery_repeat = pd.read_sql_query(query, conn)

df_first_delivery_repeat

,first_delivery_experience,customers,repeat_customers,repeat_purchase_rate_percentage
0,Late,6805,123,1.81
1,On Time / Early,77412,1513,1.95


# 6. Delivery Distance and Customer Experience

This analysis investigates whether the approximate geographic distance between sellers and customers is associated with delivery performance and customer satisfaction.

The geolocation dataset is used as a supporting geographic reference. Because multiple coordinate records can exist for the same ZIP-code prefix, representative coordinates are calculated using the mean latitude and longitude for each ZIP-code prefix.

Geographic distance is estimated using the Haversine formula and represents approximate straight-line distance rather than actual transportation route distance.

In [26]:
# Load cleaned geolocation data

geolocation_clean = pd.read_csv(
    "../data/cleaned/geolocation_clean.csv"
)

print("Geolocation data loaded successfully.")
print("Rows:", len(geolocation_clean))

geolocation_clean.head()

Geolocation data loaded successfully.
Rows: 738332


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_city_standardized
0,1037,-23.545621,-46.639292,sao paulo,SP,sao paulo
1,1046,-23.546081,-46.644820,sao paulo,SP,sao paulo
2,1046,-23.546129,-46.642951,sao paulo,SP,sao paulo
3,1041,-23.544392,-46.639499,sao paulo,SP,sao paulo
4,1035,-23.541578,-46.641607,sao paulo,SP,sao paulo


In [27]:
# Create representative coordinates for each ZIP-code prefix

zip_coordinates = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        representative_lat=("geolocation_lat", "mean"),
        representative_lng=("geolocation_lng", "mean")
    )
)

print("Representative ZIP-code locations created successfully.")
print("Unique ZIP-code prefixes:", len(zip_coordinates))

zip_coordinates.head()

Representative ZIP-code locations created successfully.
Unique ZIP-code prefixes: 19015


,geolocation_zip_code_prefix,representative_lat,representative_lng
0,1001,-23.550227,-46.634039
1,1002,-23.547657,-46.634991
2,1003,-23.549000,-46.635582
3,1004,-23.549829,-46.634792
4,1005,-23.549547,-46.636406


In [8]:
# Check customer and seller ZIP-code columns

customer_sample = pd.read_sql_query("""
SELECT
    customer_id,
    customer_zip_code_prefix
FROM customers
LIMIT 5;
""", conn)

seller_sample = pd.read_sql_query("""
SELECT
    seller_id,
    seller_zip_code_prefix
FROM sellers
LIMIT 5;
""", conn)

print("CUSTOMERS")
display(customer_sample)

print("SELLERS")
display(seller_sample)

CUSTOMERS


,customer_id,customer_zip_code_prefix
0,06b8999e2fba1a1fbc88172c00ba8bc7,14409
1,18955e83d337fd6b2def6b18a428ac77,9790
2,4e7b3e00288586ebd08712fdd0374a03,1151
3,b2b6027bc5c5109e529d4dc6358b12c3,8775
4,4f2d8ab171c80ec8364f7c12e35b23ad,13056


SELLERS


,seller_id,seller_zip_code_prefix
0,3442f8959a84dea7ee197c632cb2df15,13023
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195
4,51a04a8a6bdcb23deccc82b0b80742cf,12914


In [30]:
# Create base dataset for delivery distance analysis

distance_base = pd.read_sql_query("""
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    c.customer_zip_code_prefix,

    oi.seller_id,

    s.seller_zip_code_prefix

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN sellers s
    ON oi.seller_id = s.seller_id
""", conn)

print("Distance analysis dataset created successfully.")
print("Rows:", len(distance_base))
print("Unique orders:", distance_base["order_id"].nunique())

distance_base.head()

Distance analysis dataset created successfully.
Rows: 112650
Unique orders: 98666


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,seller_id,seller_zip_code_prefix
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,3149,3504c0cb71d7fa48d967e0e4c94d59d9,9350
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,47813,289cdb325fb7e7f891c38608bf9e0962,31570
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,75265,4869f7a5dfa277a7dca6462dcf3b52b2,14840
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,59296,66922902710d126a0e7d26b0e3805106,31842
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,9195,2c9e548be18521d1c43cde1c582c6de8,8752


In [31]:
# Attach representative coordinates for customers and sellers

# Customer coordinates
distance_analysis = distance_base.merge(
    zip_coordinates.rename(columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "representative_lat": "customer_lat",
        "representative_lng": "customer_lng"
    }),
    on="customer_zip_code_prefix",
    how="left"
)

# Seller coordinates
distance_analysis = distance_analysis.merge(
    zip_coordinates.rename(columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "representative_lat": "seller_lat",
        "representative_lng": "seller_lng"
    }),
    on="seller_zip_code_prefix",
    how="left"
)

print("Coordinates attached successfully.")
print("Rows:", len(distance_analysis))

print("\nMissing customer coordinates:",
      distance_analysis["customer_lat"].isna().sum())

print("Missing seller coordinates:",
      distance_analysis["seller_lat"].isna().sum())

distance_analysis.head()

Coordinates attached successfully.
Rows: 112650

Missing customer coordinates: 302
Missing seller coordinates: 253


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,seller_id,seller_zip_code_prefix,customer_lat,customer_lng,seller_lat,seller_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,3149,3504c0cb71d7fa48d967e0e4c94d59d9,9350,-23.577482,-46.587077,-23.680862,-46.444311
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,47813,289cdb325fb7e7f891c38608bf9e0962,31570,-12.186877,-44.540232,-19.807885,-43.980818
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,75265,4869f7a5dfa277a7dca6462dcf3b52b2,14840,-16.745150,-48.514783,-21.363473,-48.229588
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,59296,66922902710d126a0e7d26b0e3805106,31842,-5.774002,-35.270976,-19.836871,-43.923241
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,9195,2c9e548be18521d1c43cde1c582c6de8,8752,-23.676257,-46.514580,-23.541525,-46.262148


In [32]:
import numpy as np

# Keep only records with both customer and seller coordinates
distance_valid = distance_analysis.dropna(
    subset=[
        "customer_lat",
        "customer_lng",
        "seller_lat",
        "seller_lng"
    ]
).copy()

# Haversine distance calculation
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# Calculate approximate seller-to-customer distance
distance_valid["distance_km"] = haversine_distance(
    distance_valid["seller_lat"],
    distance_valid["seller_lng"],
    distance_valid["customer_lat"],
    distance_valid["customer_lng"]
)

print("Distance calculated successfully.")
print("Valid records:", len(distance_valid))
print("Missing-coordinate records excluded:", len(distance_analysis) - len(distance_valid))

distance_valid[
    [
        "order_id",
        "seller_id",
        "customer_zip_code_prefix",
        "seller_zip_code_prefix",
        "distance_km"
    ]
].head()

Distance calculated successfully.
Valid records: 112096
Missing-coordinate records excluded: 554


,order_id,seller_id,customer_zip_code_prefix,seller_zip_code_prefix,distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,3149,9350,18.538322
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,47813,31570,849.520470
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,75265,14840,514.407596
3,949d5b44dbf5de918fe9c16f97b45f8a,66922902710d126a0e7d26b0e3805106,59296,31842,1822.132331
4,ad21c59c0840e6cb83a9ceb5573f8159,2c9e548be18521d1c43cde1c582c6de8,9195,8752,29.765008


In [12]:
# Examine the distribution of estimated geographic distance

distance_summary = distance_valid["distance_km"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
)

distance_summary

count    112096.000000
mean        596.995494
std         590.039342
min           0.000000
25%         184.112924
50%         431.746959
75%         792.248222
90%        1450.776812
95%        2089.998834
max        8677.917816
Name: distance_km, dtype: float64

In [33]:
# Create interpretable distance bands

distance_bins = [0, 200, 500, 1000, 2000, float("inf")]

distance_labels = [
    "0–200 km",
    "200–500 km",
    "500–1,000 km",
    "1,000–2,000 km",
    "2,000+ km"
]

distance_valid["distance_band"] = pd.cut(
    distance_valid["distance_km"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True
)

# Check the number of records in each distance band
distance_band_counts = (
    distance_valid["distance_band"]
    .value_counts()
    .sort_index()
)

distance_band_counts

distance_band
0–200 km          28833
200–500 km        35491
500–1,000 km      30076
1,000–2,000 km    11268
2,000+ km          6428
Name: count, dtype: int64

In [35]:
# Create one approximate distance value per order

order_distance = (
    distance_valid
    .groupby("order_id", as_index=False)
    .agg(
        distance_km=("distance_km", "mean")
    )
)

# Create distance bands at the order level
order_distance["distance_band"] = pd.cut(
    order_distance["distance_km"],
    bins=[0, 200, 500, 1000, 2000, float("inf")],
    labels=[
        "0–200 km",
        "200–500 km",
        "500–1,000 km",
        "1,000–2,000 km",
        "2,000+ km"
    ],
    include_lowest=True
)

print("Order-level distance dataset created successfully.")
print("Unique orders:", len(order_distance))

order_distance.head()

Order-level distance dataset created successfully.
Unique orders: 98177


,order_id,distance_km,distance_band
0,00010242fe8c5a6d1ba2dd792cb16214,301.419473,200–500 km
1,00018f77f2f0320c557190d7a144bdd3,585.068784,"500–1,000 km"
2,000229ec398224ef6ca0657da4fc703e,312.386590,200–500 km
3,00024acbcdf0a6daa1e931b038114c75,295.481182,200–500 km
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.149244,"500–1,000 km"


In [15]:
# Load order-level delivery outcomes

order_outcomes = pd.read_sql_query("""
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM orders
WHERE order_status = 'delivered'
""", conn)

# Convert date columns to datetime
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    order_outcomes[column] = pd.to_datetime(
        order_outcomes[column],
        errors="coerce"
    )

# Join delivery outcomes with order-level distance
distance_outcomes = order_distance.merge(
    order_outcomes,
    on="order_id",
    how="inner"
)

# Keep records with all dates required for delivery analysis
distance_outcomes = distance_outcomes.dropna(
    subset=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
).copy()

# Calculate delivery time
distance_outcomes["delivery_time_days"] = (
    distance_outcomes["order_delivered_customer_date"]
    - distance_outcomes["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

# Identify late deliveries
distance_outcomes["is_late"] = (
    distance_outcomes["order_delivered_customer_date"]
    > distance_outcomes["order_estimated_delivery_date"]
).astype(int)

print("Distance and delivery outcome dataset created successfully.")
print("Valid delivered orders:", len(distance_outcomes))

distance_outcomes.head()

Distance and delivery outcome dataset created successfully.
Valid delivered orders: 95994


,order_id,distance_km,distance_band,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,301.419473,200–500 km,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,7.614421,0
1,00018f77f2f0320c557190d7a144bdd3,585.068784,"500–1,000 km",delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,16.216181,0
2,000229ec398224ef6ca0657da4fc703e,312.386590,200–500 km,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,7.948437,0
3,00024acbcdf0a6daa1e931b038114c75,295.481182,200–500 km,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,6.147269,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.149244,"500–1,000 km",delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,25.114352,0


In [41]:
# Load customer review scores

order_reviews = pd.read_sql_query("""
SELECT
    order_id,
    review_score
FROM reviews
""", conn)

print("Review records loaded:", len(order_reviews))
print("Unique orders with reviews:", order_reviews["order_id"].nunique())

order_reviews.head()

Review records loaded: 99224
Unique orders with reviews: 98673


,order_id,review_score
0,73fc7af87114b39712e6da79b0a377eb,4
1,a548910a1c6147796b98fdf73dbeba33,5
2,f9e4b658b201a9f2ecdecbb34bed034b,5
3,658677c97b385a9be170737859d3511b,5
4,8e6bfb81e283fa7e4f11123a3fb894f1,5


In [42]:
# Create one review score per order

order_review_scores = (
    order_reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean")
    )
)

print("Order-level review dataset created.")
print("Unique orders:", len(order_review_scores))

order_review_scores.head()

Order-level review dataset created.
Unique orders: 98673


,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [43]:
# Combine order-level distance with order-level review scores

distance_reviews = order_distance.merge(
    order_review_scores,
    on="order_id",
    how="inner"
)

print("Distance and review dataset created successfully.")
print("Orders with both distance and review data:", len(distance_reviews))

distance_reviews.head()

Distance and review dataset created successfully.
Orders with both distance and review data: 97430


,order_id,distance_km,distance_band,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,301.419473,200–500 km,5.0
1,00018f77f2f0320c557190d7a144bdd3,585.068784,"500–1,000 km",4.0
2,000229ec398224ef6ca0657da4fc703e,312.386590,200–500 km,5.0
3,00024acbcdf0a6daa1e931b038114c75,295.481182,200–500 km,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.149244,"500–1,000 km",5.0


In [44]:
# Analyze customer satisfaction by distance band

distance_review_summary = (
    distance_reviews
    .groupby("distance_band", observed=False)
    .agg(
        orders=("order_id", "count"),
        average_review_score=("review_score", "mean"),
        low_rated_orders=(
            "review_score",
            lambda x: ((x == 1) | (x == 2)).sum()
        ),
        low_rating_rate_percentage=(
            "review_score",
            lambda x: ((x == 1) | (x == 2)).mean() * 100
        )
    )
    .reset_index()
)

# Round values for readability
distance_review_summary["average_review_score"] = (
    distance_review_summary["average_review_score"].round(2)
)

distance_review_summary["low_rating_rate_percentage"] = (
    distance_review_summary["low_rating_rate_percentage"].round(2)
)

distance_review_summary

,distance_band,orders,average_review_score,low_rated_orders,low_rating_rate_percentage
0,0–200 km,24978,4.23,2886,11.55
1,200–500 km,30762,4.10,4447,14.46
2,"500–1,000 km",26083,4.07,3784,14.51
3,"1,000–2,000 km",9876,4.00,1596,16.16
4,"2,000+ km",5731,3.91,1063,18.55


In [20]:
# Analyze delivery performance by distance band

distance_delivery_summary = (
    distance_outcomes
    .groupby("distance_band", observed=False)
    .agg(
        orders=("order_id", "count"),
        average_delivery_time_days=("delivery_time_days", "mean"),
        late_delivery_rate_percentage=(
            "is_late",
            lambda x: x.mean() * 100
        )
    )
    .reset_index()
)

# Round for readability
distance_delivery_summary[
    "average_delivery_time_days"
] = distance_delivery_summary[
    "average_delivery_time_days"
].round(2)

distance_delivery_summary[
    "late_delivery_rate_percentage"
] = distance_delivery_summary[
    "late_delivery_rate_percentage"
].round(2)

distance_delivery_summary

,distance_band,orders,average_delivery_time_days,late_delivery_rate_percentage
0,0–200 km,24583,7.13,6.33
1,200–500 km,30311,12.11,7.37
2,"500–1,000 km",25750,14.31,8.41
3,"1,000–2,000 km",9736,17.98,10.81
4,"2,000+ km",5614,21.18,13.68


# Delivery Distance and Customer Experience

## Objective

This analysis examines whether the estimated geographic distance between sellers and customers is associated with delivery performance and customer satisfaction.

## Methodology

The geolocation dataset contains multiple latitude and longitude records for individual ZIP-code prefixes. Therefore, representative coordinates were calculated by taking the mean latitude and mean longitude for each ZIP-code prefix.

Approximate seller-to-customer geographic distance was then calculated using the Haversine formula.

Because an order may contain multiple items or sellers, seller-to-customer distances were aggregated to the order level using the mean distance across the relevant seller-item records.

The resulting distance represents an approximate straight-line geographic distance and does not represent the actual transportation or shipping route.

Orders were grouped into the following distance bands:

- 0–200 km
- 200–500 km
- 500–1,000 km
- 1,000–2,000 km
- 2,000+ km

## Key Findings

The analysis identified a consistent relationship between estimated geographic distance, delivery performance, and customer satisfaction.

As estimated distance increased:

- Average delivery time increased from approximately 7.13 days for orders within 0–200 km to 21.18 days for orders exceeding 2,000 km.
- The late-delivery rate increased from 6.33% to 13.68%.
- Average customer review scores decreased from 4.23 to 3.91.
- The proportion of low-rated orders increased from 11.55% to 18.55%.

These results indicate that longer estimated seller-to-customer distances were associated with longer delivery times, higher late-delivery rates, and lower customer satisfaction.

## Limitation

This analysis identifies associations rather than causal relationships. The calculated distance is based on representative ZIP-code-prefix coordinates and Haversine straight-line distance. It should therefore be interpreted as an approximate geographic indicator rather than the actual shipping distance or transportation route.

# Statistical Validation of Customer Experience Findings

The descriptive analysis identified differences in customer review scores across delivery-timeliness groups and estimated geographic distance bands.

This section statistically evaluates whether these observed differences are supported by the data.

Two relationships are examined:

1. Delivery delay severity and customer review score
2. Estimated geographic distance and customer review score

Because review scores are ordinal and the comparisons involve more than two independent groups, the Kruskal–Wallis test is used to evaluate whether review-score distributions differ across groups.

A statistically significant result indicates that at least one group differs from the others. It does not by itself establish causation or identify which specific groups differ.

In [13]:
from scipy.stats import kruskal

# Create order-level review scores and delivery-delay severity
query_stat = """
WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_review_score
    FROM reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
),

delivery_review AS (
    SELECT
        o.order_id,
        r.order_review_score,
        julianday(o.order_delivered_customer_date)
        - julianday(o.order_estimated_delivery_date) AS delivery_delay_days
    FROM orders o
    INNER JOIN order_reviews r
        ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)

SELECT
    order_id,
    order_review_score,
    delivery_delay_days,

    CASE
        WHEN delivery_delay_days <= 0
            THEN 'On Time or Early'
        WHEN delivery_delay_days BETWEEN 1 AND 7
            THEN '1-7 Days Late'
        WHEN delivery_delay_days BETWEEN 8 AND 14
            THEN '8-14 Days Late'
        ELSE '15+ Days Late'
    END AS delay_severity

FROM delivery_review
"""

delay_review_data = pd.read_sql_query(query_stat, conn)

delay_review_data.head()

,order_id,order_review_score,delivery_delay_days,delay_severity
0,e481f51cbdc54678b7cc49136f2d6af7,4.0,-7.107488,On Time or Early
1,53cdb2fc8bc7dce0b6741e2150273451,4.0,-5.355729,On Time or Early
2,47770eb9100c2d0c44946d9cf07ec65d,5.0,-17.245498,On Time or Early
3,949d5b44dbf5de918fe9c16f97b45f8a,5.0,-12.980069,On Time or Early
4,ad21c59c0840e6cb83a9ceb5573f8159,5.0,-9.238171,On Time or Early


In [8]:
print("order_level:", "order_level" in globals())
print("delivery_delay_review:", "delivery_delay_review" in globals())
print("pd:", "pd" in globals())

order_level: False
delivery_delay_review: False
pd: True


In [9]:
print("conn:", "conn" in globals())
print("df_delay_reviews:", "df_delay_reviews" in globals())

conn: True
df_delay_reviews: False


In [14]:
# Prepare the groups for the Kruskal-Wallis test

groups = [
    group["order_review_score"].values
    for _, group in delay_review_data.groupby(
        "delay_severity",
        observed=True
    )
]

# Run Kruskal-Wallis test
kruskal_result = kruskal(*groups)

print("Kruskal-Wallis H-statistic:", kruskal_result.statistic)
print("p-value:", kruskal_result.pvalue)
print("Sample size:", len(delay_review_data))

print("\nGroup sizes:")
print(
    delay_review_data
    .groupby("delay_severity", observed=True)["order_review_score"]
    .count()
)

Kruskal-Wallis H-statistic: 8637.20451081741
p-value: 0.0
Sample size: 95824

Group sizes:
delay_severity
1-7 Days Late        3129
15+ Days Late        3255
8-14 Days Late       1277
On Time or Early    88163
Name: order_review_score, dtype: int64


In [15]:
# Calculate epsilon-squared effect size for the Kruskal-Wallis test

k = len(groups)
n = len(delay_review_data)
H = kruskal_result.statistic

epsilon_squared = (H - k + 1) / (n - k)

print("Epsilon-squared effect size:", epsilon_squared)

Epsilon-squared effect size: 0.09010858391585691


In [18]:
import numpy as np

In [19]:
# 95% bootstrap confidence intervals for median review score

rng = np.random.default_rng(42)

bootstrap_results = []

for severity, group in delay_review_data.groupby(
    "delay_severity",
    observed=True
):
    scores = group["order_review_score"].dropna().to_numpy()
    n_group = len(scores)

    bootstrap_medians = []

    for _ in range(2000):
        sample = rng.choice(
            scores,
            size=n_group,
            replace=True
        )
        bootstrap_medians.append(np.median(sample))

    ci_lower = np.percentile(bootstrap_medians, 2.5)
    ci_upper = np.percentile(bootstrap_medians, 97.5)

    bootstrap_results.append({
        "delay_severity": severity,
        "n": n_group,
        "median_review_score": np.median(scores),
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper
    })

delay_ci = pd.DataFrame(bootstrap_results)

delay_ci

,delay_severity,n,median_review_score,ci_lower_95,ci_upper_95
0,1-7 Days Late,3129,3.0,3.0,3.0
1,15+ Days Late,3255,2.0,2.0,2.0
2,8-14 Days Late,1277,1.0,1.0,1.0
3,On Time or Early,88163,5.0,5.0,5.0


In [20]:
from scipy.stats import mannwhitneyu
from itertools import combinations

# Define the delivery-delay groups
group_names = [
    "On Time or Early",
    "1-7 Days Late",
    "8-14 Days Late",
    "15+ Days Late"
]

pairwise_results = []

for group_a, group_b in combinations(group_names, 2):

    scores_a = delay_review_data.loc[
        delay_review_data["delay_severity"] == group_a,
        "order_review_score"
    ].dropna()

    scores_b = delay_review_data.loc[
        delay_review_data["delay_severity"] == group_b,
        "order_review_score"
    ].dropna()

    statistic, p_value = mannwhitneyu(
        scores_a,
        scores_b,
        alternative="two-sided"
    )

    pairwise_results.append({
        "group_a": group_a,
        "group_b": group_b,
        "n_a": len(scores_a),
        "n_b": len(scores_b),
        "u_statistic": statistic,
        "raw_p_value": p_value
    })

pairwise_results = pd.DataFrame(pairwise_results)

# Holm-Bonferroni correction
pairwise_results = pairwise_results.sort_values(
    "raw_p_value"
).reset_index(drop=True)

m = len(pairwise_results)

pairwise_results["holm_p_value"] = [
    min((m - i) * p, 1.0)
    for i, p in enumerate(pairwise_results["raw_p_value"])
]

# Make sure adjusted p-values are monotonic
for i in range(1, m):
    pairwise_results.loc[i, "holm_p_value"] = max(
        pairwise_results.loc[i, "holm_p_value"],
        pairwise_results.loc[i - 1, "holm_p_value"]
    )

pairwise_results

,group_a,group_b,n_a,n_b,u_statistic,raw_p_value,holm_p_value
0,On Time or Early,1-7 Days Late,88163,3129,205097556.0,0.000000e+00,0.000000e+00
1,On Time or Early,8-14 Days Late,88163,1277,101068793.0,0.000000e+00,0.000000e+00
2,On Time or Early,15+ Days Late,88163,3255,218569568.5,0.000000e+00,0.000000e+00
3,1-7 Days Late,8-14 Days Late,3129,1277,2757246.0,4.893398e-99,1.468019e-98
4,8-14 Days Late,15+ Days Late,1277,3255,1429680.0,2.420896e-70,4.841793e-70
5,1-7 Days Late,15+ Days Late,3129,3255,5392612.5,1.889764e-05,1.889764e-05


## Statistical Finding: Delivery Delay and Customer Satisfaction

The Kruskal–Wallis test identified statistically significant differences in customer review-score distributions across the four delivery-delay severity groups (H = 8,637.20, p < 0.001, n = 95,824).

The epsilon-squared effect size was 0.0901, indicating that delivery-delay severity was associated with a measurable share of the variation in review-score distributions.

Pairwise Mann–Whitney U tests with Holm correction showed statistically significant differences between all six pairs of delivery-delay groups (all adjusted p < 0.001).

The median review score also showed a clear difference across delivery-delay severity:

- On Time or Early: 5.0
- 1–7 Days Late: 3.0
- 8–14 Days Late: 1.0
- 15+ Days Late: 2.0

The 95% bootstrap confidence intervals for the median were stable at the observed medians for all four groups.

### SO WHAT?

The analysis indicates that delivery timeliness is strongly associated with customer satisfaction in the Olist dataset. Orders delivered on time or early had substantially higher median review scores, while orders experiencing delivery delays had lower review scores.

This suggests that delivery performance is an important operational area for customer-experience improvement. Monitoring late-delivery rates and investigating the causes of severe delays may therefore help identify opportunities to improve customer satisfaction.

### Limitation

This analysis identifies an association rather than a causal relationship. Other factors, such as product characteristics, seller performance, order value, geography, or customer expectations, may also influence review scores.

In [21]:
print("distance_reviews:", "distance_reviews" in globals())

if "distance_reviews" in globals():
    print("Rows:", len(distance_reviews))
    print("Columns:", distance_reviews.columns.tolist())

distance_reviews: False


In [45]:
# Statistical Validation: Geographic Distance vs Customer Review Score

from scipy.stats import kruskal

# Remove records with missing review scores or distance bands
distance_stat_data = distance_reviews.dropna(
    subset=["distance_band", "review_score"]
).copy()

# Prepare review-score groups by distance band
distance_groups = [
    group["review_score"].values
    for _, group in distance_stat_data.groupby(
        "distance_band",
        observed=True
    )
]

# Run Kruskal-Wallis test
distance_kruskal_result = kruskal(*distance_groups)

print("Kruskal-Wallis H-statistic:",
      distance_kruskal_result.statistic)

print("p-value:",
      distance_kruskal_result.pvalue)

print("Sample size:",
      len(distance_stat_data))

print("\nGroup sizes:")

print(
    distance_stat_data
    .groupby("distance_band", observed=True)["review_score"]
    .count()
)

Kruskal-Wallis H-statistic: 431.14977112208817
p-value: 5.159704826965628e-92
Sample size: 97430

Group sizes:
distance_band
0–200 km          24978
200–500 km        30762
500–1,000 km      26083
1,000–2,000 km     9876
2,000+ km          5731
Name: review_score, dtype: int64


In [46]:
# Effect size: Epsilon-squared for Kruskal-Wallis

n = len(distance_stat_data)
k = distance_stat_data["distance_band"].nunique()
H = distance_kruskal_result.statistic

epsilon_squared_distance = (H - k + 1) / (n - k)

print("Epsilon-squared:", epsilon_squared_distance)

Epsilon-squared: 0.004384395905795106


In [47]:
# Pairwise Mann-Whitney U tests with Holm correction

from scipy.stats import mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests

distance_groups_dict = {
    band: group["review_score"].values
    for band, group in distance_stat_data.groupby(
        "distance_band",
        observed=True
    )
}

pairwise_results = []

for group1, group2 in combinations(distance_groups_dict.keys(), 2):
    
    stat, p_value = mannwhitneyu(
        distance_groups_dict[group1],
        distance_groups_dict[group2],
        alternative="two-sided"
    )
    
    pairwise_results.append({
        "group_1": group1,
        "group_2": group2,
        "U_statistic": stat,
        "raw_p_value": p_value
    })

pairwise_results_df = pd.DataFrame(pairwise_results)

# Holm correction
reject, corrected_p, _, _ = multipletests(
    pairwise_results_df["raw_p_value"],
    method="holm"
)

pairwise_results_df["holm_p_value"] = corrected_p
pairwise_results_df["significant"] = reject

display(pairwise_results_df)

,group_1,group_2,U_statistic,raw_p_value,holm_p_value,significant
0,0–200 km,200–500 km,401414514.5,3.300191e-25,2.310134e-24,True
1,0–200 km,"500–1,000 km",347154536.5,8.372869e-48,6.698296e-47,True
2,0–200 km,"1,000–2,000 km",134710622.5,1.794688e-52,1.615219e-51,True
3,0–200 km,"2,000+ km",80138416.0,2.292115e-58,2.292115e-57,True
4,200–500 km,"500–1,000 km",409228487.0,3.934953e-06,1.180486e-05,True
5,200–500 km,"1,000–2,000 km",158933915.0,9.384796e-15,4.692398e-14,True
6,200–500 km,"2,000+ km",94686580.0,1.773163e-23,1.063898e-22,True
7,"500–1,000 km","1,000–2,000 km",132212996.0,1.700113e-05,3.400226e-05,True
8,"500–1,000 km","2,000+ km",78859077.0,4.718627e-13,1.887451e-12,True
9,"1,000–2,000 km","2,000+ km",29118781.0,9.681044e-04,9.681044e-04,True


## Statistical Finding: Geographic Distance and Customer Satisfaction

A Kruskal–Wallis test was used to examine whether customer review scores differed across the five estimated geographic distance bands.

The test showed a statistically significant difference in review-score distributions across distance groups (H = 431.15, p < 0.001, n = 97,430).

The epsilon-squared effect size was 0.0044, indicating a small overall effect.

Pairwise Mann–Whitney U tests with Holm correction showed statistically significant differences between all 10 pairs of distance bands (all adjusted p < 0.001).

Descriptively, average review scores decreased from 4.23 for orders estimated at 0–200 km to 3.91 for orders estimated at 2,000+ km, while the low-rating rate increased from 11.55% to 18.55%.

### SO WHAT?

The results indicate that estimated geographic distance is associated with customer satisfaction. However, the small effect size indicates that distance alone explains a limited amount of variation in review scores. Distance should therefore be treated as a supporting operational signal rather than a standalone explanation of customer dissatisfaction.

### Limitation

Distance is estimated using representative coordinates for customer and seller ZIP-code prefixes and represents approximate straight-line geographic distance rather than actual transportation-route distance. The analysis identifies association, not causation.

## 8. Customer Voice & Review Theme Analysis

Customer reviews provide direct qualitative evidence about the purchasing experience.

This section analyzes review text to identify recurring customer sentiment, complaint themes, and potential operational issues.

The analysis will focus on reviews containing usable text. Reviews without text will remain part of the overall review dataset but will be excluded from text-based analysis.

The objective is to identify recurring customer concerns and use them to generate hypotheses that can later be tested against the transactional and operational data.

AI-based analysis will be treated as a hypothesis-generation tool rather than as proof of causation.

In [50]:
# Restore cleaned reviews dataset after kernel restart

reviews_clean = pd.read_csv("../data/cleaned/reviews_clean.csv")

print("reviews_clean restored successfully.")
print("Rows:", len(reviews_clean))
print("Columns:", reviews_clean.columns.tolist())

reviews_clean restored successfully.
Rows: 99224
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [51]:
# ------------------------------------------------------------
# Customer Voice: Review Text Availability
# ------------------------------------------------------------

reviews_text = reviews_clean[
    [
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
].copy()

# Combine title and message into one text field
reviews_text["review_text"] = (
    reviews_text["review_comment_title"].fillna("").str.strip()
    + " "
    + reviews_text["review_comment_message"].fillna("").str.strip()
).str.strip()

# Identify reviews with usable text
reviews_text["has_review_text"] = reviews_text["review_text"].ne("")

print("Total review records:", len(reviews_text))
print(
    "Reviews with usable text:",
    reviews_text["has_review_text"].sum()
)
print(
    "Reviews without usable text:",
    (~reviews_text["has_review_text"]).sum()
)

print("\nReview text availability:")
display(
    reviews_text["has_review_text"]
    .value_counts()
    .rename(index={
        True: "Has text",
        False: "No text"
    })
)

print("\nSample review text:")
display(
    reviews_text.loc[
        reviews_text["has_review_text"],
        ["order_id", "review_score", "review_text"]
    ].head(10)
)

Total review records: 99224
Reviews with usable text: 42687
Reviews without usable text: 56537

Review text availability:


has_review_text
No text     56537
Has text    42687
Name: count, dtype: int64


Sample review text:


,order_id,review_score,review_text
3,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
4,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...
9,b9bf720beb4ab3728760088589c62129,4,recomendo aparelho eficiente. no site a marca ...
12,9d6f15f95d01e79bd1349cc208361f09,4,"Mas um pouco ,travando...pelo valor ta Boa."
15,e51478e7e277a83743b6f9991dbfa3fb,5,"Super recomendo Vendedor confiável, produto ok..."
16,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E..."
19,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto Péssimo
22,4fc44d78867142c627497b60a7e0228a,5,Ótimo Loja nota 10
24,79832b7cb59ac6f887088ffd686e1d5e,5,obrigado pela atençao amim dispensada
27,2ca73e2ff9e3a186ad1e1ffb9b1d9c10,5,A compra foi realizada facilmente.\r\nA entreg...


In [52]:
# ------------------------------------------------------------
# Review Text Quality Check
# ------------------------------------------------------------

review_text_data = reviews_text[
    reviews_text["has_review_text"]
].copy()

# Basic text-length metrics
review_text_data["character_count"] = (
    review_text_data["review_text"].str.len()
)

review_text_data["word_count"] = (
    review_text_data["review_text"]
    .str.split()
    .str.len()
)

print("Usable review texts:", len(review_text_data))

print("\nCharacter count:")
print(review_text_data["character_count"].describe())

print("\nWord count:")
print(review_text_data["word_count"].describe())

print("\nReview score distribution among text reviews:")
display(
    review_text_data["review_score"]
    .value_counts()
    .sort_index()
    .rename_axis("review_score")
    .reset_index(name="review_count")
)

print("\nShortest review texts:")
display(
    review_text_data[
        ["review_score", "review_text", "character_count", "word_count"]
    ]
    .sort_values("character_count")
    .head(15)
)

Usable review texts: 42687

Character count:
count    42687.000000
mean        69.035514
std         54.922285
min          1.000000
25%         27.000000
50%         53.000000
75%         95.000000
max        229.000000
Name: character_count, dtype: float64

Word count:
count    42687.000000
mean        11.718650
std          9.688165
min          1.000000
25%          4.000000
50%          9.000000
75%         16.000000
max         48.000000
Name: word_count, dtype: float64

Review score distribution among text reviews:


,review_score,review_count
0,1,8829
1,2,2165
2,3,3643
3,4,6274
4,5,21776



Shortest review texts:


,review_score,review_text,character_count,word_count
49002,4,8,1,1
4575,1,8,1,1
62778,3,3,1,1
27041,1,.,1,1
34584,4,?,1,1
42131,5,.,1,1
598,4,4,1,1
65441,5,9,1,1
28716,4,7,1,1
583,4,5,1,1


## 7. Order Value and Customer Satisfaction

This analysis examines whether customer satisfaction varies across different order-value ranges.

Order value is defined as the total product price plus freight value associated with an order.

Review scores are aggregated to the order level so that each order contributes one observation to the analysis.

The analysis compares customer review scores and low-rating rates across order-value bands.

This analysis identifies associations between order value and customer satisfaction; it does not establish causation.

In [53]:
# ------------------------------------------------------------
# Order Value vs Customer Satisfaction
# ------------------------------------------------------------

order_value_reviews = pd.read_sql_query("""
WITH order_value AS (
    SELECT
        oi.order_id,
        SUM(oi.price + oi.freight_value) AS order_value
    FROM order_items oi
    GROUP BY oi.order_id
),

order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score
    FROM reviews
    WHERE review_score IS NOT NULL
    GROUP BY order_id
)

SELECT
    ov.order_id,
    ov.order_value,
    r.review_score
FROM order_value ov
INNER JOIN order_reviews r
    ON ov.order_id = r.order_id
WHERE ov.order_value IS NOT NULL
  AND r.review_score IS NOT NULL
""", conn)

print("Order value and review dataset created successfully.")
print("Orders:", len(order_value_reviews))

print("\nSample:")
display(order_value_reviews.head())

Order value and review dataset created successfully.
Orders: 97917

Sample:


,order_id,order_value,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,5.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,4.0
2,000229ec398224ef6ca0657da4fc703e,216.87,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,5.0


In [54]:
print("Order value statistics:")
display(order_value_reviews["order_value"].describe())

print("\nOrder value percentiles:")
print(
    order_value_reviews["order_value"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

Order value statistics:


count    97917.000000
mean       160.338303
std        219.268269
min          9.590000
25%         61.870000
50%        105.280000
75%        176.740000
max      13664.080000
Name: order_value, dtype: float64


Order value percentiles:
0.25      61.8700
0.50     105.2800
0.75     176.7400
0.90     307.3960
0.95     450.1120
0.99    1058.6712
Name: order_value, dtype: float64


In [55]:
order_value_reviews["order_value_band"] = pd.cut(
    order_value_reviews["order_value"],
    bins=[-np.inf, 61.87, 105.28, 176.74, 307.40, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
        "Premium"
    ],
    include_lowest=True
)

order_value_summary = (
    order_value_reviews
    .groupby("order_value_band", observed=False)
    .agg(
        orders=("order_id", "count"),
        average_order_value=("order_value", "mean"),
        average_review_score=("review_score", "mean"),
        low_rated_orders=("review_score", lambda x: (x <= 2).sum())
    )
    .reset_index()
)

order_value_summary["low_rating_rate_percentage"] = (
    order_value_summary["low_rated_orders"]
    / order_value_summary["orders"]
    * 100
)

display(order_value_summary)

,order_value_band,orders,average_order_value,average_review_score,low_rated_orders,low_rating_rate_percentage
0,Low,24481,42.917623,4.193974,2844,11.617173
1,Medium,24506,81.660153,4.131335,3260,13.302865
2,High,24452,137.224180,4.107067,3441,14.072469
3,Very High,14686,224.963438,4.011984,2459,16.743838
4,Premium,9792,611.600987,3.951542,1845,18.841912


In [56]:
from scipy.stats import kruskal

groups = [
    group["review_score"].values
    for _, group in order_value_reviews.groupby(
        "order_value_band",
        observed=False
    )
]

kruskal_result = kruskal(*groups)

print("Kruskal-Wallis Test: Order Value vs Review Score")
print("H-statistic:", kruskal_result.statistic)
print("p-value:", kruskal_result.pvalue)
print("Sample size:", len(order_value_reviews))

Kruskal-Wallis Test: Order Value vs Review Score
H-statistic: 176.1475497467719
p-value: 5.009512098942263e-37
Sample size: 97917


In [57]:
n = len(order_value_reviews)
k = order_value_reviews["order_value_band"].nunique()

epsilon_squared = (
    (kruskal_result.statistic - k + 1)
    / (n - k)
)

print("Epsilon-squared:", epsilon_squared)

Epsilon-squared: 0.0017581864301288083


In [58]:
from itertools import combinations
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

band_order = [
    "Low",
    "Medium",
    "High",
    "Very High",
    "Premium"
]

pairwise_results = []

for band1, band2 in combinations(band_order, 2):
    group1 = order_value_reviews.loc[
        order_value_reviews["order_value_band"] == band1,
        "review_score"
    ]

    group2 = order_value_reviews.loc[
        order_value_reviews["order_value_band"] == band2,
        "review_score"
    ]

    u_stat, p_value = mannwhitneyu(
        group1,
        group2,
        alternative="two-sided"
    )

    pairwise_results.append({
        "comparison": f"{band1} vs {band2}",
        "u_statistic": u_stat,
        "raw_p_value": p_value
    })

pairwise_results = pd.DataFrame(pairwise_results)

# Holm correction for multiple comparisons
reject, corrected_p, _, _ = multipletests(
    pairwise_results["raw_p_value"],
    method="holm"
)

pairwise_results["holm_adjusted_p_value"] = corrected_p
pairwise_results["significant"] = reject

display(pairwise_results)

,comparison,u_statistic,raw_p_value,holm_adjusted_p_value,significant
0,Low vs Medium,306372726.0,3.760961e-06,1.128288e-05,True
1,Low vs High,307320837.0,7.008282e-09,3.504141e-08,True
2,Low vs Very High,189601984.0,1.632280e-24,1.469052e-23,True
3,Low vs Premium,128012941.5,1.264112e-28,1.264112e-27,True
4,Medium vs High,301258291.5,2.367623e-01,2.367623e-01,False
5,Medium vs Very High,185988454.5,4.704356e-10,2.822613e-09,True
6,Medium vs Premium,125632717.0,2.374356e-14,1.899485e-13,True
7,High vs Very High,184577892.0,2.126206e-07,8.504824e-07,True
8,High vs Premium,124697078.0,1.712680e-11,1.198876e-10,True
9,Very High vs Premium,72903885.0,4.055849e-02,8.111697e-02,False


### Statistical Finding: Order Value and Customer Satisfaction

The Kruskal–Wallis test identified a statistically significant difference in review-score distributions across order-value bands (H = 176.15, p < 0.001, n = 97,917).

However, the epsilon-squared effect size was very small (ε² = 0.00176), indicating that order-value band explains only a small portion of the variation in review-score distributions.

Pairwise Mann–Whitney U tests with Holm correction showed significant differences for most order-value-band comparisons. The Medium vs High comparison was not statistically significant (adjusted p = 0.237), and the Very High vs Premium comparison was also not statistically significant (adjusted p = 0.081).

Descriptively, average review scores decreased from 4.19 in the Low order-value band to 3.95 in the Premium band, while the low-rating rate increased from 11.62% to 18.84%.

**SO WHAT:** Higher-value orders show somewhat lower customer satisfaction in this dataset, but the very small effect size indicates that order value is not a strong standalone explanation for poor customer experience. Operational factors such as delivery performance, seller performance, product category, freight costs, and geography should therefore be considered alongside order value.

**Limitation:** This analysis identifies an association between order value and customer satisfaction and does not establish causation. Other customer, product, seller, payment, and delivery characteristics may contribute to review outcomes.

In [61]:
# Restore order_level from the cleaned source tables

orders_clean = pd.read_csv(
    "../data/cleaned/orders_clean.csv"
)

payments_clean = pd.read_csv(
    "../data/cleaned/payments_clean.csv"
)

reviews_clean = pd.read_csv(
    "../data/cleaned/reviews_clean.csv"
)

order_items_clean = pd.read_csv(
    "../data/cleaned/order_items_clean.csv"
)

# Payment summary
payment_summary = (
    payments_clean
    .groupby("order_id")
    .agg(
        payment_total=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

# Review summary
review_summary = (
    reviews_clean
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_score", "count")
    )
    .reset_index()
)

# Build order-level dataset
order_level = orders_clean.merge(
    payment_summary,
    on="order_id",
    how="left"
)

order_level = order_level.merge(
    review_summary,
    on="order_id",
    how="left"
)

print("Base order_level restored:", order_level.shape)

Base order_level restored: (99441, 13)


In [63]:
order_item_summary = (
    order_items_clean
    .groupby("order_id")
    .agg(
        total_product_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count")
    )
    .reset_index()
)

order_item_summary["freight_ratio"] = (
    order_item_summary["total_freight_value"]
    / order_item_summary["total_product_price"]
)

order_level = order_level.merge(
    order_item_summary,
    on="order_id",
    how="left"
)

print("Freight metrics restored successfully.")
print("order_level shape:", order_level.shape)
print("Orders with freight ratio:", order_level["freight_ratio"].notna().sum())

Freight metrics restored successfully.
order_level shape: (99441, 17)
Orders with freight ratio: 98666


In [64]:
freight_reviews = order_level[
    [
        "order_id",
        "total_product_price",
        "total_freight_value",
        "freight_ratio",
        "avg_review_score"
    ]
].dropna(
    subset=["freight_ratio", "avg_review_score"]
).copy()

print("Freight ratio and review dataset created successfully.")
print("Orders:", len(freight_reviews))

print("\nFreight ratio statistics:")
display(freight_reviews["freight_ratio"].describe())

print("\nFreight ratio percentiles:")
print(
    freight_reviews["freight_ratio"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

Freight ratio and review dataset created successfully.
Orders: 97917

Freight ratio statistics:


count    97917.000000
mean         0.308431
std          0.314860
min          0.000000
25%          0.131938
50%          0.224374
75%          0.380196
max         21.447059
Name: freight_ratio, dtype: float64


Freight ratio percentiles:
0.25    0.131938
0.50    0.224374
0.75    0.380196
0.90    0.622556
0.95    0.839355
0.99    1.461670
Name: freight_ratio, dtype: float64


In [65]:
order_value = freight_reviews["freight_ratio"]

freight_reviews["freight_ratio_band"] = pd.cut(
    freight_reviews["freight_ratio"],
    bins=[-np.inf, 0.131938, 0.224374, 0.380196, 0.622556, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
        "Extreme"
    ],
    include_lowest=True
)

freight_summary = (
    freight_reviews
    .groupby("freight_ratio_band", observed=False)
    .agg(
        orders=("order_id", "count"),
        average_freight_ratio=("freight_ratio", "mean"),
        average_review_score=("avg_review_score", "mean"),
        low_rated_orders=("avg_review_score", lambda x: (x <= 2).sum())
    )
    .reset_index()
)

freight_summary["low_rating_rate_percentage"] = (
    freight_summary["low_rated_orders"]
    / freight_summary["orders"]
    * 100
)

display(freight_summary)

,freight_ratio_band,orders,average_freight_ratio,average_review_score,low_rated_orders,low_rating_rate_percentage
0,Low,24478,0.083868,4.135407,3445,14.073862
1,Medium,24571,0.175415,4.117408,3352,13.642098
2,High,24388,0.291829,4.099906,3465,14.207807
3,Very High,14684,0.481116,4.084571,2100,14.301280
4,Extreme,9796,0.985688,4.041752,1487,15.179665


In [66]:
from scipy.stats import kruskal

groups = [
    group["avg_review_score"].values
    for _, group in freight_reviews.groupby(
        "freight_ratio_band",
        observed=False
    )
]

freight_kruskal = kruskal(*groups)

print("Kruskal-Wallis Test: Freight Ratio vs Review Score")
print("H-statistic:", freight_kruskal.statistic)
print("p-value:", freight_kruskal.pvalue)
print("Sample size:", len(freight_reviews))

Kruskal-Wallis Test: Freight Ratio vs Review Score
H-statistic: 68.95494579517587
p-value: 3.772041372848179e-14
Sample size: 97917


In [67]:
n = len(freight_reviews)
k = freight_reviews["freight_ratio_band"].nunique()

freight_epsilon_squared = (
    (freight_kruskal.statistic - k + 1)
    / (n - k)
)

print("Epsilon-squared:", freight_epsilon_squared)

Epsilon-squared: 0.0006634012766073196


In [68]:
from itertools import combinations
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

band_order = [
    "Low",
    "Medium",
    "High",
    "Very High",
    "Extreme"
]

freight_pairwise_results = []

for band1, band2 in combinations(band_order, 2):

    group1 = freight_reviews.loc[
        freight_reviews["freight_ratio_band"] == band1,
        "avg_review_score"
    ]

    group2 = freight_reviews.loc[
        freight_reviews["freight_ratio_band"] == band2,
        "avg_review_score"
    ]

    u_stat, p_value = mannwhitneyu(
        group1,
        group2,
        alternative="two-sided"
    )

    freight_pairwise_results.append({
        "comparison": f"{band1} vs {band2}",
        "u_statistic": u_stat,
        "raw_p_value": p_value
    })

freight_pairwise_results = pd.DataFrame(
    freight_pairwise_results
)

reject, corrected_p, _, _ = multipletests(
    freight_pairwise_results["raw_p_value"],
    method="holm"
)

freight_pairwise_results["holm_adjusted_p_value"] = corrected_p
freight_pairwise_results["significant"] = reject

display(freight_pairwise_results)

,comparison,u_statistic,raw_p_value,holm_adjusted_p_value,significant
0,Low vs Medium,306305397.0,5.916482e-05,2.958241e-04,True
1,Low vs High,304773960.5,5.320943e-06,3.724660e-05,True
2,Low vs Very High,185093587.0,2.159119e-08,1.943207e-07,True
3,Low vs Extreme,125462603.0,3.348104e-14,3.348104e-13,True
4,Medium vs High,300442605.5,5.549540e-01,5.549540e-01,False
5,Medium vs Very High,182516090.5,2.943582e-02,8.830747e-02,False
6,Medium vs Extreme,123793948.0,3.647550e-06,2.918040e-05,True
7,High vs Very High,180645002.5,1.000276e-01,2.000552e-01,False
8,High vs Extreme,122514481.5,3.440499e-05,2.064299e-04,True
9,Very High vs Extreme,73134307.0,1.295004e-02,5.180018e-02,False


### Statistical Finding: Freight Burden and Customer Satisfaction

The Kruskal–Wallis test identified statistically significant differences in review-score distributions across freight-ratio bands (H = 68.95, p < 0.001, n = 97,917).

However, the overall effect size was extremely small (ε² = 0.00066), indicating that freight-ratio bands account for only a very small portion of the variation in review-score distributions.

Pairwise Mann–Whitney U tests with Holm correction showed significant differences for:
- Low vs Medium
- Low vs High
- Low vs Very High
- Low vs Extreme
- Medium vs Extreme
- High vs Extreme

The following comparisons were not statistically significant after Holm correction:
- Medium vs High
- Medium vs Very High
- High vs Very High
- Very High vs Extreme

Descriptively, average review score declined gradually from 4.14 in the Low freight-ratio band to 4.04 in the Extreme band. However, the low-rating rate remained relatively stable, increasing only from 14.07% to 15.18%.

**SO WHAT:** Freight burden shows a statistically detectable but very small association with customer satisfaction. It should therefore be treated as a supporting customer-experience signal rather than a major standalone explanation for poor reviews. Delivery performance, seller performance, product/category factors and other operational variables should be considered alongside freight burden.

**Limitation:** This analysis identifies association, not causation. Freight ratio may also reflect product size, product type, shipment characteristics, distance and other operational factors.